In [1]:
!pip install selenium beautifulsoup4 pandas requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 3.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.6/82.6 KB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.7/512.7 KB 1.9 MB/s eta 0:00:00a 0:00:01
  Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
  Using cached h11-0.16.0-py3-none-any.whl (37 kB)


In [2]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

base_url = "https://www.greenclimate.fund/boardroom/documents"
filter_param = "f[]=field_subtype:292"  # Action item filter

documents = []

# Compile regex patterns for funding proposal-related keywords and specific funding packages
general_pattern = re.compile(r'funding proposal|proposal|funding request', re.IGNORECASE)

# Pattern for funding packages FP01 to FP275 (with optional leading zeros)
# This matches "FP" followed by 1-3 digits, where the number is between 1 and 275
funding_package_pattern = re.compile(r'FP(\d{1,3})', re.IGNORECASE)

def is_valid_funding_package(fp_number):
    """Check if the FP number is within the valid range (1-275)"""
    try:
        num = int(fp_number)
        return 1 <= num <= 275
    except ValueError:
        return False

def extract_fp_number(title):
    """Extract FP number from title if it exists and is valid"""
    match = funding_package_pattern.search(title)
    if match:
        fp_num = match.group(1)
        if is_valid_funding_package(fp_num):
            return f"FP{fp_num.zfill(3)}"  # Format as FP001, FP034, etc.
    return None

for page in range(40):  # Pages 0 to 39 inclusive
    url = f"{base_url}?{filter_param}&page={page}"
    print(f"Fetching page {page} ...")

    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to retrieve page {page}, status code: {response.status_code}")
        break

    soup = BeautifulSoup(response.text, 'html.parser')

    # Select all rows in the documents table body
    rows = soup.select('table.views-table tbody tr')

    if not rows:
        print(f"No documents found on page {page}, stopping.")
        break

    for row in rows:
        try:
            ref_tag = row.select_one('.views-field-field-code')
            title_tag = row.select_one('.views-field-title a')
            type_tag = row.select_one('.views-field-field-subtype')

            if ref_tag and title_tag and type_tag:
                ref = ref_tag.get_text(strip=True)
                title = title_tag.get_text(strip=True)
                link = 'https://www.greenclimate.fund' + title_tag['href']
                doc_type = type_tag.get_text(strip=True)

                # First check: general funding proposal keywords
                if general_pattern.search(title):
                    # Second check: specific funding package pattern (FP01-FP275)
                    fp_number = extract_fp_number(title)
                    if fp_number:
                        documents.append({
                            'Ref #': ref,
                            'Title': title,
                            'Link': link,
                            'Type': doc_type,
                            'FP Number': fp_number
                        })
                        print(f"Found funding package: {fp_number} - {title}")
                    else:
                        # Include general funding proposals that don't have specific FP numbers
                        documents.append({
                            'Ref #': ref,
                            'Title': title,
                            'Link': link,
                            'Type': doc_type,
                            'FP Number': 'N/A'
                        })
        except Exception as e:
            print(f"Error parsing row: {e}")
            continue

    # Be polite and don't hammer the server
    time.sleep(1)

# Create DataFrame from collected documents
df = pd.DataFrame(documents)

print(f"\nTotal funding proposal related documents scraped: {len(df)}")

# Separate funding packages from general proposals
funding_packages = df[df['FP Number'] != 'N/A']
general_proposals = df[df['FP Number'] == 'N/A']

print(f"Specific funding packages (FP01-FP275): {len(funding_packages)}")
print(f"General funding proposals: {len(general_proposals)}")

# Display funding packages first
if len(funding_packages) > 0:
    print("\nFunding Packages Found:")
    print(funding_packages[['FP Number', 'Title', 'Ref #']].head(10))

# Display general proposals
if len(general_proposals) > 0:
    print("\nGeneral Funding Proposals:")
    print(general_proposals[['Title', 'Ref #']].head(10))

# Optionally save to CSV
df.to_csv('gcf_funding_proposals.csv', index=False)
funding_packages.to_csv('gcf_funding_packages.csv', index=False)

Fetching page 0 ...
Found funding package: FP264 - Consideration of funding proposals – Addendum XXIX Funding proposal package for FP264
Found funding package: FP230 - Status of approved funding proposals: adding host countries in respect of FP230 (“Kuali Fund-GCF”)
Found funding package: FP224 - Consideration of funding proposals: Extension of deadline in respect of FP224 (Renewstable Barbados Project)
Found funding package: FP242 - Consideration of funding proposals: extension of the deadline in respect of FP242 “Caribbean Net-Zero and Resilient Private Sector”
Fetching page 1 ...
Fetching page 2 ...
Found funding package: FP221 - Consideration of funding proposals: extension of deadline in respect of FP221 (Rwanda Green Investment Facility (RGIF))
Found funding package: FP205 - Status of approved funding proposals: adding host countries in respect of FP205 (“Infrastructure Climate Resilient Fund”)
Found funding package: FP269 - Consideration of funding proposals – Addendum XI Fundin

In [9]:
len(funding_packages)

299

In [3]:
# Iterate over the rows of the funding_packages DataFrame
for index, row in funding_packages.iterrows():
    print(f"Ref #: {row['Ref #']}")
    print(f"Title: {row['Title']}")
    print(f"Link: {row['Link']}")
    print(f"Type: {row['Type']}")
    print(f"FP Number: {row['FP Number']}")
    print("-" * 20) # Print a separator line for clarity

Ref #: GCF/B.43/02/Add.29
Title: Consideration of funding proposals – Addendum XXIX Funding proposal package for FP264
Link: https://www.greenclimate.fund/document/gcf-b43-02-add29
Type: Action item
FP Number: FP264
--------------------
Ref #: GCF/B.43/16
Title: Status of approved funding proposals: adding host countries in respect of FP230 (“Kuali Fund-GCF”)
Link: https://www.greenclimate.fund/document/gcf-b43-16
Type: Action item
FP Number: FP230
--------------------
Ref #: GCF/B.43/14
Title: Consideration of funding proposals: Extension of deadline in respect of FP224 (Renewstable Barbados Project)
Link: https://www.greenclimate.fund/document/gcf-b43-14
Type: Action item
FP Number: FP224
--------------------
Ref #: GCF/B.43/09
Title: Consideration of funding proposals: extension of the deadline in respect of FP242 “Caribbean Net-Zero and Resilient Private Sector”
Link: https://www.greenclimate.fund/document/gcf-b43-09
Type: Action item
FP Number: FP242
--------------------
Ref #: GC

In [4]:
def order_funding_packages(funding_packages_df):
    """
    Take a funding packages dataframe and return an ordered dictionary
    of funding packages in descending order (highest FP number first).

    Args:
        funding_packages_df: DataFrame containing funding packages with 'FP Number' column

    Returns:
        OrderedDict: {fp_number: row_data} ordered from highest to lowest FP number
    """
    from collections import OrderedDict

    # Filter out rows where FP Number is 'N/A' or empty
    valid_packages = funding_packages_df[
        (funding_packages_df['FP Number'] != 'N/A') &
        (funding_packages_df['FP Number'].notna())
    ].copy()

    if len(valid_packages) == 0:
        return OrderedDict()

    # Extract numeric part from FP numbers for sorting
    def extract_fp_numeric(fp_str):
        try:
            # Remove 'FP' prefix and convert to int
            return int(fp_str.replace('FP', ''))
        except (ValueError, AttributeError):
            return 0

    # Add a temporary column for sorting
    valid_packages['sort_key'] = valid_packages['FP Number'].apply(extract_fp_numeric)

    # Sort by numeric value in descending order
    sorted_packages = valid_packages.sort_values('sort_key', ascending=False)

    # Create ordered dictionary
    ordered_packages = OrderedDict()

    for _, row in sorted_packages.iterrows():
        fp_number = row['FP Number']
        # Convert row to dict and remove the temporary sort_key
        row_dict = row.to_dict()
        row_dict.pop('sort_key', None)
        ordered_packages[fp_number] = row_dict

    return ordered_packages

# Example usage:
ordered_fp = order_funding_packages(funding_packages)
#
# Print the ordered funding packages
#for fp_num, data in ordered_fp.items():
 #    print(f"{fp_num}:TITLE: {data['Title']} , LINK: {data['Link']} ")

 # Or get just the FP numbers in descending order
fp_numbers_desc = list(ordered_fp.keys())
len(ordered_fp)
#print(f"FP numbers in descending order: {fp_numbers_desc}")

273

# Task
Modify the provided Python code to download PDF files asynchronously using `asyncio` and `aiohttp`.

## Install aiohttp

### Subtask:
Add a cell to install the aiohttp library.


**Reasoning**:
The subtask is to install the `aiohttp` library. This requires using pip in a new code cell.



In [5]:
!pip install aiohttp
!pip install aiofiles

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 2.6 MB/s eta 0:00:00a 0:00:010m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.7/241.7 KB 103.0 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.5/219.5 KB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.0/347.0 KB 79.7 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.9/196.9 KB 1.3 MB/s eta 0:00:00a 0:00:01


## Modify the download function

### Subtask:
Rewrite the `download_pdf_from_doc_page` function to be asynchronous using `aiohttp`.


**Reasoning**:
Rewrite the `download_pdf_from_doc_page` function to be asynchronous using `aiohttp`.



In [6]:
import aiohttp
import os
from bs4 import BeautifulSoup
import asyncio
import aiofiles # Import aiofiles

async def download_pdf_from_doc_page_async(session, doc_url, save_folder='pdfs', semaphore=None, idx=None):
    """
    Given a document page URL, download the PDF linked on that page asynchronously,
    with an optional semaphore to limit concurrency and optional prefix index.
    """

    full_url = doc_url
    base_site = 'https://www.greenclimate.fund'

    async with semaphore: # Acquire semaphore
        try:
            async with session.get(full_url) as response:
                response.raise_for_status()
                html_content = await response.text()
        except aiohttp.ClientConnectorError as e:
            print(f"Connection Error loading document page {full_url}: {e}")
            return False
        except aiohttp.ClientResponseError as e:
            print(f"HTTP Error loading document page {full_url}: {e.status}")
            return False
        except Exception as e:
            print(f"An unexpected error occurred loading document page {full_url}: {e}")
            return False


        soup = BeautifulSoup(html_content, 'html.parser')

        # Find all <a> tags with href ending with .pdf
        pdf_links = [a['href'] for a in soup.find_all('a', href=True) if a['href'].lower().endswith('.pdf')]

        if len(pdf_links) < 1:
            print(f"No PDF links found on page {full_url}")
            return False

        # Here, just take the first PDF link
        pdf_url = pdf_links[0]
        if not pdf_url.startswith('http'):
            pdf_url = base_site + pdf_url

        print(f"Downloading PDF from: {pdf_url}")

        try:
            async with session.get(pdf_url) as pdf_response:
                pdf_response.raise_for_status()

                # Create folder if not exists
                os.makedirs(save_folder, exist_ok=True)

                # Extract filename from URL
                filename = pdf_url.split('/')[-1]
                # Prefix filename with index, padded to 2 digits
                if idx is not None:
                    filename = f"{str(idx).zfill(2)}_{filename}"
                filepath = os.path.join(save_folder, filename)

                async with aiofiles.open(filepath, 'wb') as afile:
                    await afile.write(await pdf_response.read())

                print(f"Saved PDF to {filepath}")
                return True

        except aiohttp.ClientConnectorError as e:
            print(f"Connection Error downloading PDF {pdf_url}: {e}")
            return False
        except aiohttp.ClientResponseError as e:
            print(f"HTTP Error downloading PDF {pdf_url}: {e.status}")
            return False
        except Exception as e:
            print(f"An unexpected error occurred downloading PDF {pdf_url}: {e}")
            return False

## Create and run asynchronous tasks

### Subtask:
Modify the loop to create asynchronous tasks for each download and run them concurrently using `asyncio.gather`.


**Reasoning**:
Define an asynchronous main function to orchestrate the concurrent download of PDFs using asyncio and aiohttp.



In [7]:
import aiohttp
import os
from bs4 import BeautifulSoup
import aiofiles

async def main_async(value_item_dict:dict)->None:
    """
    Orchestrates the asynchronous download of PDF documents with limited concurrency.
    """
    save_folder = 'pdfs_20_2'
    os.makedirs(save_folder, exist_ok=True)

    # Create a semaphore to limit concurrency to 4 tasks
    semaphore = asyncio.Semaphore(4)

    async with aiohttp.ClientSession() as session:
        tasks = []

        for idx, document in enumerate(value_item_dict, start=1):
            print(f" start downloading content index: {idx}")
            doc_url = document['Link']
            # Pass idx to prefix the file
            task = asyncio.create_task(
                download_pdf_from_doc_page_async(session, doc_url, save_folder, semaphore, idx)
            )
            tasks.append(task)
        await asyncio.gather(*tasks)


**Reasoning**:
Execute the asynchronous main function to start the concurrent download process.



In [8]:
import asyncio
import aiohttp
import os # Import os if not already imported

len(funding_packages)
# Assuming 'documents' is the list containing document dictionaries from the previous step
print(funding_packages.to_dict(orient='index'))
#for key , value in enumerate(funding_packages.to_dict(orient='index').items()):
   # print(key, value  )


len(funding_packages.to_dict(orient='index'))

# Simply await the main_async function at the top level
await main_async(ordered_fp.values())

{5: {'Ref #': 'GCF/B.43/02/Add.29', 'Title': 'Consideration of funding proposals – Addendum XXIX Funding proposal package for FP264', 'Link': 'https://www.greenclimate.fund/document/gcf-b43-02-add29', 'Type': 'Action item', 'FP Number': 'FP264'}, 6: {'Ref #': 'GCF/B.43/16', 'Title': 'Status of approved funding proposals: adding host countries in respect of FP230 (“Kuali Fund-GCF”)', 'Link': 'https://www.greenclimate.fund/document/gcf-b43-16', 'Type': 'Action item', 'FP Number': 'FP230'}, 7: {'Ref #': 'GCF/B.43/14', 'Title': 'Consideration of funding proposals: Extension of deadline in respect of FP224 (Renewstable Barbados Project)', 'Link': 'https://www.greenclimate.fund/document/gcf-b43-14', 'Type': 'Action item', 'FP Number': 'FP224'}, 16: {'Ref #': 'GCF/B.43/09', 'Title': 'Consideration of funding proposals: extension of the deadline in respect of FP242 “Caribbean Net-Zero and Resilient Private Sector”', 'Link': 'https://www.greenclimate.fund/document/gcf-b43-09', 'Type': 'Action i

In [9]:
import os

def count_pdf_files(folder_path='pdfs_20'):
    """Counts the number of PDF files in a given folder."""
    count = 0
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            if filename.lower().endswith('.pdf'):
                count += 1
    return count

pdf_count = count_pdf_files(folder_path='pdfs_20')
print(f"Number of PDF files in the 'pdfs' folder: {pdf_count}")

Number of PDF files in the 'pdfs' folder: 0


In [ ]:
!pip install openai langchain langchain-community faiss-cpu pdfplumber numpy chainlit


In [ ]:
# Dependencies (install via pip)
# pip install openai langchain faiss-cpu pdfplumber numpy chainlit

import os
import glob
import faiss
import pickle
import openai
from langchain.embeddings import OpenAIEmbeddings
import numpy as np

# Set your Gemini model (ensure your environment is configured for Google Gemini)
GEMINI_MODEL = "gemini-pro"

# 1. Parse PDF directly with Gemini
def parse_pdf_with_gemini(pdf_path: str) -> str:
    """Send a PDF file to Gemini and receive a structured text representation with metadata."""
    with open(pdf_path, 'rb') as f:
        pdf_data = f.read()
    response = openai.ChatCompletion.create(
        model=GEMINI_MODEL,
        messages=[
            {"role": "user", "content": (
                "You are a document intelligence assistant.\n"
                "Parse the attached PDF and generate a .txt output that includes:\n"
                "- A document title\n"
                "- Metadata (author, date, page count if available)\n"
                "- Structured headings and sections\n"
                "- Full narrative text under each heading\n"
                "Respond only with the appropriately formatted text."
            )}
        ],
        files=[
            {"filename": os.path.basename(pdf_path), "content": pdf_data}
        ],
        temperature=0.0,
        max_tokens=16384
    )
    return response.choices[0].message.content

# 2. Semantic chunking
def chunk_text(full_text: str, chunk_size: int = 1000, chunk_overlap: int = 200) -> list[str]:
    """Split text into semantically coherent chunks."""
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    return splitter.split_text(full_text)

# 3. Embedding and indexing
def build_faiss_index(chunks: list[str], embedding_model=None):
    """Embed chunks and build a FAISS index. Returns index and metadata list."""
    if embedding_model is None:
        embedding_model = OpenAIEmbeddings()
    embeddings = embedding_model.embed_documents(chunks)
    dim = len(embeddings[0])
    index = faiss.IndexFlatL2(dim)
    index.add(np.array(embeddings, dtype="float32"))
    metadata = chunks.copy()
    return index, metadata

# 4. Save and load index+metadata
def save_index(index, metadata, index_path: str, meta_path: str):
    faiss.write_index(index, index_path)
    with open(meta_path, "wb") as f:
        pickle.dump(metadata, f)

def load_index(index_path: str, meta_path: str):
    index = faiss.read_index(index_path)
    with open(meta_path, "rb") as f:
        metadata = pickle.load(f)
    return index, metadata

# 5. Query function
def query_index(query: str, index, metadata: list[str], embedding_model=None, top_k: int = 5) -> list[str]:
    """Retrieve top-k chunks relevant to the query."""
    if embedding_model is None:
        embedding_model = OpenAIEmbeddings()
    q_emb = embedding_model.embed_query(query)
    D, I = index.search(np.array([q_emb], dtype="float32"), top_k)
    return [metadata[i] for i in I[0]]

# 6. End-to-end ingestion pipeline
def ingest_and_index(pdf_folder: str, index_path: str, meta_path: str):
    """Process all PDFs: parse via Gemini, chunk, and build + save FAISS index."""
    all_chunks = []
    for pdf_file in glob.glob(os.path.join(pdf_folder, "*.pdf")):
        structured_text = parse_pdf_with_gemini(pdf_file)
        chunks = chunk_text(structured_text)
        all_chunks.extend(chunks)
    index, metadata = build_faiss_index(all_chunks)
    save_index(index, metadata, index_path, meta_path)


In [ ]:
!pip install google-generativeai langchain faiss-cpu pdfplumber numpy chainlit

In [10]:
GEMINI_API_KEY="REDACTED-GOOGLE-KEY"

In [ ]:
# Dependencies (install via pip)
# pip install google-generativeai langchain faiss-cpu pdfplumber numpy chainlit aiofiles

import os
import glob
import faiss
import pickle
import asyncio
import aiofiles
import google.generativeai as genai
import numpy as np
from langchain.text_splitter import RecursiveCharacterTextSplitter
from typing import List, Tuple, Optional
from concurrent.futures import ThreadPoolExecutor
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- Configuration ---
genai.configure(api_key=f"{GEMINI_API_KEY}")

PARSING_MODEL = "gemini-2.5-flash-latest"
EMBEDDING_MODEL = "text-embedding-004"

# Configuration for async processing
MAX_CONCURRENT_PDFS = 3  # Adjust based on API limits
MAX_CONCURRENT_CHUNKS = 50  # For embedding batches
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
CACHE_FOLDER = "txt_cache"  # Folder to store parsed text files

class AsyncPDFRAG:
    def __init__(self, max_concurrent_pdfs: int = MAX_CONCURRENT_PDFS,
                 max_concurrent_chunks: int = MAX_CONCURRENT_CHUNKS,
                 cache_folder: str = CACHE_FOLDER):
        self.max_concurrent_pdfs = max_concurrent_pdfs
        self.max_concurrent_chunks = max_concurrent_chunks
        self.cache_folder = cache_folder
        self.parsing_model = genai.GenerativeModel(PARSING_MODEL)
        self.executor = ThreadPoolExecutor(max_workers=max_concurrent_pdfs)

        # Create cache folder if it doesn't exist
        os.makedirs(self.cache_folder, exist_ok=True)

    def get_cache_path(self, pdf_path: str) -> str:
        """Generate cache file path for a given PDF."""
        pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
        return os.path.join(self.cache_folder, f"{pdf_name}.txt")

    async def is_pdf_cached(self, pdf_path: str) -> bool:
        """Check if PDF has already been parsed and cached."""
        cache_path = self.get_cache_path(pdf_path)
        if not os.path.exists(cache_path):
            return False

        # Check if cache is newer than the PDF file
        try:
            pdf_mtime = os.path.getmtime(pdf_path)
            cache_mtime = os.path.getmtime(cache_path)
            return cache_mtime > pdf_mtime
        except OSError:
            return False

    async def load_from_cache(self, pdf_path: str) -> Optional[str]:
        """Load parsed text from cache if available."""
        cache_path = self.get_cache_path(pdf_path)
        try:
            async with aiofiles.open(cache_path, 'r', encoding='utf-8') as f:
                content = await f.read()
                logger.info(f"Loaded cached text for {os.path.basename(pdf_path)}")
                return content
        except Exception as e:
            logger.error(f"Error loading cache for {pdf_path}: {e}")
            return None

    async def save_to_cache(self, pdf_path: str, content: str):
        """Save parsed text to cache."""
        cache_path = self.get_cache_path(pdf_path)
        try:
            async with aiofiles.open(cache_path, 'w', encoding='utf-8') as f:
                await f.write(content)
                logger.info(f"Cached parsed text for {os.path.basename(pdf_path)}")
        except Exception as e:
            logger.error(f"Error saving cache for {pdf_path}: {e}")

    async def parse_pdf_with_gemini(self, pdf_path: str) -> str:
        """
        Asynchronously parse a PDF file using Gemini 2.5 Pro with caching.
        """
        # Check if already cached
        if await self.is_pdf_cached(pdf_path):
            cached_content = await self.load_from_cache(pdf_path)
            if cached_content:
                return cached_content

        logger.info(f"Parsing {os.path.basename(pdf_path)} with Gemini 2.5 Pro...")

        def _upload_and_parse():
            try:
                uploaded_file = genai.upload_file(
                    path=pdf_path,
                    display_name=os.path.basename(pdf_path)
                )

                prompt = (
                    "You are a document intelligence assistant. Your task is to analyze the attached PDF file.\n"
                    "Please parse the document and generate a clean .txt output that includes the following:\n"
                    "- A clear document title.\n"
                    "- Key metadata such as author, publication date, and page count, if available.\n"
                    "- Structured headings and subheadings to preserve the document's layout.\n"
                    "- The full narrative text under each corresponding heading.\n"
                    "Ensure your response contains only the structured text output, ready for saving to a file."
                )

                response = self.parsing_model.generate_content([prompt, uploaded_file])
                return response.text
            except Exception as e:
                logger.error(f"Error parsing PDF {pdf_path}: {e}")
                return ""

        # Run the blocking operation in a thread pool
        loop = asyncio.get_event_loop()
        parsed_text = await loop.run_in_executor(self.executor, _upload_and_parse)

        # Cache the result if parsing was successful
        if parsed_text:
            await self.save_to_cache(pdf_path, parsed_text)

        return parsed_text

    def chunk_text(self, full_text: str) -> List[str]:
        """Split text into semantically coherent chunks."""
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", " ", ""],
            length_function=len
        )
        return splitter.split_text(full_text)

    async def embed_chunks_batch(self, chunks: List[str]) -> np.ndarray:
        """
        Embed chunks in batches asynchronously.
        """
        def _embed_batch(chunk_batch):
            try:
                result = genai.embed_content(
                    model=EMBEDDING_MODEL,
                    content=chunk_batch,
                    task_type="retrieval_document"
                )
                return result["embedding"]
            except Exception as e:
                logger.error(f"Error embedding batch: {e}")
                return []

        # Split chunks into batches to avoid API limits
        batch_size = min(self.max_concurrent_chunks, len(chunks))
        batches = [chunks[i:i + batch_size] for i in range(0, len(chunks), batch_size)]

        logger.info(f"Embedding {len(chunks)} chunks in {len(batches)} batches...")

        # Run batches concurrently
        loop = asyncio.get_event_loop()
        tasks = [loop.run_in_executor(self.executor, _embed_batch, batch) for batch in batches]

        embeddings_batches = await asyncio.gather(*tasks)

        # Flatten the results
        all_embeddings = []
        for batch_embeddings in embeddings_batches:
            if batch_embeddings:
                all_embeddings.extend(batch_embeddings)

        return np.array(all_embeddings, dtype="float32")

    async def build_faiss_index(self, chunks: List[str]) -> Tuple[faiss.Index, List[str]]:
        """Build FAISS index asynchronously."""
        if not chunks:
            raise ValueError("No chunks provided for indexing")

        embeddings_np = await self.embed_chunks_batch(chunks)

        if len(embeddings_np) == 0:
            raise ValueError("Failed to generate embeddings")

        # Build the FAISS index
        dim = embeddings_np.shape[1]
        index = faiss.IndexFlatL2(dim)
        index.add(embeddings_np)

        logger.info("FAISS index built successfully.")
        return index, chunks.copy()

    async def save_index(self, index: faiss.Index, metadata: List[str],
                        index_path: str, meta_path: str):
        """Save FAISS index and metadata asynchronously."""
        def _save_index():
            faiss.write_index(index, index_path)

        def _save_metadata():
            with open(meta_path, "wb") as f:
                pickle.dump(metadata, f)

        loop = asyncio.get_event_loop()
        await asyncio.gather(
            loop.run_in_executor(self.executor, _save_index),
            loop.run_in_executor(self.executor, _save_metadata)
        )

        logger.info(f"Index saved to {index_path} and metadata to {meta_path}")

    async def load_index(self, index_path: str, meta_path: str) -> Tuple[faiss.Index, List[str]]:
        """Load FAISS index and metadata asynchronously."""
        def _load_index():
            return faiss.read_index(index_path)

        def _load_metadata():
            with open(meta_path, "rb") as f:
                return pickle.load(f)

        loop = asyncio.get_event_loop()
        index, metadata = await asyncio.gather(
            loop.run_in_executor(self.executor, _load_index),
            loop.run_in_executor(self.executor, _load_metadata)
        )

        logger.info("Index and metadata loaded successfully.")
        return index, metadata

    async def query_index(self, query: str, index: faiss.Index,
                         metadata: List[str], top_k: int = 5) -> List[str]:
        """Query the index asynchronously."""
        def _embed_query():
            return genai.embed_content(
                model=EMBEDDING_MODEL,
                content=query,
                task_type="retrieval_query"
            )["embedding"]

        def _search_index(q_emb):
            distances, indices = index.search(np.array([q_emb], dtype="float32"), top_k)
            return [metadata[i] for i in indices[0]]

        loop = asyncio.get_event_loop()
        q_emb = await loop.run_in_executor(self.executor, _embed_query)
        results = await loop.run_in_executor(self.executor, _search_index, q_emb)

        return results

    async def process_pdf_batch(self, pdf_files: List[str]) -> List[str]:
        """Process multiple PDFs concurrently with caching support."""
        semaphore = asyncio.Semaphore(self.max_concurrent_pdfs)

        async def _process_single_pdf(pdf_file):
            async with semaphore:
                # Check cache first
                if await self.is_pdf_cached(pdf_file):
                    cached_content = await self.load_from_cache(pdf_file)
                    if cached_content:
                        return self.chunk_text(cached_content)

                # Parse if not cached
                structured_text = await self.parse_pdf_with_gemini(pdf_file)
                if structured_text:
                    return self.chunk_text(structured_text)
                return []

        # Process PDFs concurrently
        tasks = [_process_single_pdf(pdf_file) for pdf_file in pdf_files]
        results = await asyncio.gather(*tasks, return_exceptions=True)

        # Flatten results and handle exceptions
        all_chunks = []
        processed_count = 0
        cached_count = 0

        for i, result in enumerate(results):
            if isinstance(result, Exception):
                logger.error(f"Error processing PDF {pdf_files[i]}: {result}")
            elif result:
                all_chunks.extend(result)
                processed_count += 1
                # Check if it was from cache
                if await self.is_pdf_cached(pdf_files[i]):
                    cached_count += 1

        logger.info(f"Processed {processed_count} PDFs ({cached_count} from cache, {processed_count - cached_count} newly parsed)")
        return all_chunks

    async def ingest_and_index(self, pdf_folder: str, index_path: str, meta_path: str):
        """
        Complete ingestion pipeline: process PDFs, chunk, embed, and save index.
        """
        pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))
        if not pdf_files:
            logger.warning(f"No PDF files found in '{pdf_folder}'.")
            return

        logger.info(f"Processing {len(pdf_files)} PDF files...")

        # Process all PDFs concurrently
        all_chunks = await self.process_pdf_batch(pdf_files)

        if all_chunks:
            logger.info(f"Generated {len(all_chunks)} chunks from {len(pdf_files)} PDFs")
            index, metadata = await self.build_faiss_index(all_chunks)
            await self.save_index(index, metadata, index_path, meta_path)
        else:
            logger.warning("No text was extracted from the PDFs. Index not built.")

    async def clear_cache(self):
        """Clear all cached text files."""
        try:
            import shutil
            if os.path.exists(self.cache_folder):
                shutil.rmtree(self.cache_folder)
                os.makedirs(self.cache_folder, exist_ok=True)
                logger.info("Cache cleared successfully")
        except Exception as e:
            logger.error(f"Error clearing cache: {e}")

    async def get_cache_stats(self) -> dict:
        """Get statistics about cached files."""
        try:
            cache_files = glob.glob(os.path.join(self.cache_folder, "*.txt"))
            total_size = sum(os.path.getsize(f) for f in cache_files)
            return {
                "cached_files": len(cache_files),
                "total_size_mb": total_size / (1024 * 1024),
                "cache_folder": self.cache_folder
            }
        except Exception as e:
            logger.error(f"Error getting cache stats: {e}")
    def __del__(self):
        """Cleanup executor on destruction."""
        if hasattr(self, 'executor'):
            self.executor.shutdown(wait=True)


# --- Async Chainlit Integration ---
# Example of how to structure the Chainlit part with async support

"""
import chainlit as cl

# Global variables
rag_system = None
index = None
metadata = None

@cl.on_chat_start
async def start():
    global rag_system, index, metadata

    try:
        rag_system = AsyncPDFRAG()
        index, metadata = await rag_system.load_index("my_document_index.faiss", "my_document_meta.pkl")
        await cl.Message(content="Hello! I'm ready to answer questions about your documents.").send()
    except FileNotFoundError:
        await cl.Message(content="The document index is not available. Please run the ingestion script.").send()
    except Exception as e:
        await cl.Message(content=f"Error loading index: {e}").send()

@cl.on_message
async def main(message: cl.Message):
    global rag_system, index, metadata

    if not index or not rag_system:
        await cl.Message(content="Sorry, I cannot answer questions without the document index.").send()
        return

    query = message.content

    try:
        # Retrieve relevant context from the FAISS index
        hits = await rag_system.query_index(query, index, metadata)
        context = "\n\n".join(hits)

        # Create a prompt for the RAG model
        prompt = (
            "You are an expert Q&A assistant who answers questions based on the provided context.\n"
            "Use ONLY the following context to answer the question. If the answer is not in the context, say you don't know.\n\n"
            f"Context:\n---\n{context}\n---\n\n"
            f"Question: {query}\n"
            "Answer:"
        )

        # Generate the answer asynchronously
        def _generate_response():
            return rag_system.parsing_model.generate_content(prompt)

        loop = asyncio.get_event_loop()
        response = await loop.run_in_executor(rag_system.executor, _generate_response)

        await cl.Message(content=response.text).send()

    except Exception as e:
        await cl.Message(content=f"Error processing query: {e}").send()
"""

# --- Example Usage ---
async def main():
    # Configuration
    PDF_FOLDER_PATH = '/content/pdfs_20'
    INDEX_FILE_PATH = 'my_document_index.faiss'
    METADATA_FILE_PATH = 'document_meta_.pkl'
    CACHE_FOLDER = 'txt_cache_2'  # Custom cache folder

    # Create the PDF folder if it doesn't exist
    if not os.path.exists(PDF_FOLDER_PATH):
        os.makedirs(PDF_FOLDER_PATH)
        logger.info(f"Created folder '{PDF_FOLDER_PATH}'. Please add your PDF files there.")
        return

    # Initialize the RAG system with custom cache folder
    rag_system = AsyncPDFRAG(cache_folder=CACHE_FOLDER)

    try:
        # Show cache statistics
        cache_stats = await rag_system.get_cache_stats()
        logger.info(f"Cache stats: {cache_stats}")

        # Run the ingestion and indexing pipeline
        await rag_system.ingest_and_index(PDF_FOLDER_PATH, INDEX_FILE_PATH, METADATA_FILE_PATH)

        # Show updated cache statistics
        cache_stats = await rag_system.get_cache_stats()
        logger.info(f"Updated cache stats: {cache_stats}")

        # Example query after indexing
        if os.path.exists(INDEX_FILE_PATH):
            index, metadata = await rag_system.load_index(INDEX_FILE_PATH, METADATA_FILE_PATH)

            # Test query
            test_query = "What is the main topic of the documents?"
            results = await rag_system.query_index(test_query, index, metadata)

            logger.info(f"Query: {test_query}")
            logger.info(f"Results: {len(results)} chunks retrieved")

    except Exception as e:
        logger.error(f"Error in main pipeline: {e}")
    finally:
        # Cleanup is handled automatically by __del__
        pass

if __name__ == '__main__':
    asyncio.run(main())

In [ ]:
# Dependencies (install via pip)
# pip install google-generativeai langchain faiss-cpu pdfplumber numpy chainlit aiofiles

import os
import glob
import faiss
import pickle
import asyncio
import aiofiles
import google.generativeai as genai
import numpy as np
from langchain.text_splitter import RecursiveCharacterTextSplitter
from typing import List, Tuple, Optional
from concurrent.futures import ThreadPoolExecutor
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- Configuration ---
genai.configure(api_key=f"{GEMINI_API_KEY}")

PARSING_MODEL = "gemini-2.5-flash-latest"
EMBEDDING_MODEL = "text-embedding-004"

# Configuration for async processing
MAX_CONCURRENT_PDFS = 3  # Adjust based on API limits
MAX_CONCURRENT_CHUNKS = 50  # For embedding batches
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
CACHE_FOLDER = "txt_cache"  # Folder to store parsed text files

class AsyncPDFRAG:
    def __init__(self, max_concurrent_pdfs: int = MAX_CONCURRENT_PDFS,
                 max_concurrent_chunks: int = MAX_CONCURRENT_CHUNKS,
                 cache_folder: str = CACHE_FOLDER):
        self.max_concurrent_pdfs = max_concurrent_pdfs
        self.max_concurrent_chunks = max_concurrent_chunks
        self.cache_folder = cache_folder
        self.parsing_model = genai.GenerativeModel(PARSING_MODEL)
        self.executor = ThreadPoolExecutor(max_workers=max_concurrent_pdfs)

        # Create cache folder if it doesn't exist
        os.makedirs(self.cache_folder, exist_ok=True)

    def get_cache_path(self, pdf_path: str) -> str:
        """Generate cache file path for a given PDF."""
        pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
        return os.path.join(self.cache_folder, f"{pdf_name}.txt")

    async def is_pdf_cached(self, pdf_path: str) -> bool:
        """Check if PDF has already been parsed and cached."""
        cache_path = self.get_cache_path(pdf_path)
        if not os.path.exists(cache_path):
            return False

        # Check if cache is newer than the PDF file
        try:
            pdf_mtime = os.path.getmtime(pdf_path)
            cache_mtime = os.path.getmtime(cache_path)
            return cache_mtime > pdf_mtime
        except OSError:
            return False

    async def load_from_cache(self, pdf_path: str) -> Optional[str]:
        """Load parsed text from cache if available."""
        cache_path = self.get_cache_path(pdf_path)
        try:
            async with aiofiles.open(cache_path, 'r', encoding='utf-8') as f:
                content = await f.read()
                logger.info(f"Loaded cached text for {os.path.basename(pdf_path)}")
                return content
        except Exception as e:
            logger.error(f"Error loading cache for {pdf_path}: {e}")
            return None

    async def save_to_cache(self, pdf_path: str, content: str):
        """Save parsed text to cache."""
        cache_path = self.get_cache_path(pdf_path)
        try:
            async with aiofiles.open(cache_path, 'w', encoding='utf-8') as f:
                await f.write(content)
                logger.info(f"Cached parsed text for {os.path.basename(pdf_path)}")
        except Exception as e:
            logger.error(f"Error saving cache for {pdf_path}: {e}")

    async def parse_pdf_with_gemini(self, pdf_path: str) -> str:
        """
        Asynchronously parse a PDF file using Gemini 2.5 Pro with caching.
        """
        # Check if already cached
        if await self.is_pdf_cached(pdf_path):
            cached_content = await self.load_from_cache(pdf_path)
            if cached_content:
                return cached_content

        logger.info(f"Parsing {os.path.basename(pdf_path)} with Gemini 2.5 Pro...")

        def _upload_and_parse():
            try:
                uploaded_file = genai.upload_file(
                    path=pdf_path,
                    display_name=os.path.basename(pdf_path)
                )

                prompt = (
                    "You are a document intelligence assistant. Your task is to analyze the attached PDF file.\n"
                    "Please parse the document and generate a clean .txt output that includes the following:\n"
                    "- A clear document title.\n"
                    "- Key metadata such as author, publication date, and page count, if available.\n"
                    "- Structured headings and subheadings to preserve the document's layout.\n"
                    "- The full narrative text under each corresponding heading.\n"
                    "Ensure your response contains only the structured text output, ready for saving to a file."
                )

                response = self.parsing_model.generate_content([prompt, uploaded_file])
                return response.text
            except Exception as e:
                logger.error(f"Error parsing PDF {pdf_path}: {e}")
                return ""

        # Run the blocking operation in a thread pool
        loop = asyncio.get_event_loop()
        parsed_text = await loop.run_in_executor(self.executor, _upload_and_parse)

        # Cache the result if parsing was successful
        if parsed_text:
            await self.save_to_cache(pdf_path, parsed_text)

        return parsed_text

    def chunk_text(self, full_text: str) -> List[str]:
        """Split text into semantically coherent chunks."""
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", " ", ""],
            length_function=len
        )
        return splitter.split_text(full_text)

    async def embed_chunks_batch(self, chunks: List[str]) -> np.ndarray:
        """
        Embed chunks in batches asynchronously.
        """
        def _embed_batch(chunk_batch):
            try:
                result = genai.embed_content(
                    model=EMBEDDING_MODEL,
                    content=chunk_batch,
                    task_type="retrieval_document"
                )
                return result["embedding"]
            except Exception as e:
                logger.error(f"Error embedding batch: {e}")
                return []

        # Split chunks into batches to avoid API limits
        batch_size = min(self.max_concurrent_chunks, len(chunks))
        batches = [chunks[i:i + batch_size] for i in range(0, len(chunks), batch_size)]

        logger.info(f"Embedding {len(chunks)} chunks in {len(batches)} batches...")

        # Run batches concurrently
        loop = asyncio.get_event_loop()
        tasks = [loop.run_in_executor(self.executor, _embed_batch, batch) for batch in batches]

        embeddings_batches = await asyncio.gather(*tasks)

        # Flatten the results
        all_embeddings = []
        for batch_embeddings in embeddings_batches:
            if batch_embeddings:
                all_embeddings.extend(batch_embeddings)

        return np.array(all_embeddings, dtype="float32")

    async def build_faiss_index(self, chunks: List[str]) -> Tuple[faiss.Index, List[str]]:
        """Build FAISS index asynchronously."""
        if not chunks:
            raise ValueError("No chunks provided for indexing")

        embeddings_np = await self.embed_chunks_batch(chunks)

        if len(embeddings_np) == 0:
            raise ValueError("Failed to generate embeddings")

        # Build the FAISS index
        dim = embeddings_np.shape[1]
        index = faiss.IndexFlatL2(dim)
        index.add(embeddings_np)

        logger.info("FAISS index built successfully.")
        return index, chunks.copy()

    async def save_index(self, index: faiss.Index, metadata: List[str],
                        index_path: str, meta_path: str):
        """Save FAISS index and metadata asynchronously."""
        def _save_index():
            faiss.write_index(index, index_path)

        def _save_metadata():
            with open(meta_path, "wb") as f:
                pickle.dump(metadata, f)

        loop = asyncio.get_event_loop()
        await asyncio.gather(
            loop.run_in_executor(self.executor, _save_index),
            loop.run_in_executor(self.executor, _save_metadata)
        )

        logger.info(f"Index saved to {index_path} and metadata to {meta_path}")

    async def load_index(self, index_path: str, meta_path: str) -> Tuple[faiss.Index, List[str]]:
        """Load FAISS index and metadata asynchronously."""
        def _load_index():
            return faiss.read_index(index_path)

        def _load_metadata():
            with open(meta_path, "rb") as f:
                return pickle.load(f)

        loop = asyncio.get_event_loop()
        index, metadata = await asyncio.gather(
            loop.run_in_executor(self.executor, _load_index),
            loop.run_in_executor(self.executor, _load_metadata)
        )

        logger.info("Index and metadata loaded successfully.")
        return index, metadata

    async def query_index(self, query: str, index: faiss.Index,
                         metadata: List[str], top_k: int = 5) -> List[str]:
        """Query the index asynchronously."""
        def _embed_query():
            return genai.embed_content(
                model=EMBEDDING_MODEL,
                content=query,
                task_type="retrieval_query"
            )["embedding"]

        def _search_index(q_emb):
            distances, indices = index.search(np.array([q_emb], dtype="float32"), top_k)
            return [metadata[i] for i in indices[0]]

        loop = asyncio.get_event_loop()
        q_emb = await loop.run_in_executor(self.executor, _embed_query)
        results = await loop.run_in_executor(self.executor, _search_index, q_emb)

        return results

    async def process_pdf_batch(self, pdf_files: List[str]) -> List[str]:
        """Process multiple PDFs concurrently with caching support."""
        semaphore = asyncio.Semaphore(self.max_concurrent_pdfs)

        async def _process_single_pdf(pdf_file):
            async with semaphore:
                # Check cache first
                if await self.is_pdf_cached(pdf_file):
                    cached_content = await self.load_from_cache(pdf_file)
                    if cached_content:
                        return self.chunk_text(cached_content)

                # Parse if not cached
                structured_text = await self.parse_pdf_with_gemini(pdf_file)
                if structured_text:
                    return self.chunk_text(structured_text)
                return []

        # Process PDFs concurrently
        tasks = [_process_single_pdf(pdf_file) for pdf_file in pdf_files]
        results = await asyncio.gather(*tasks, return_exceptions=True)

        # Flatten results and handle exceptions
        all_chunks = []
        processed_count = 0
        cached_count = 0

        for i, result in enumerate(results):
            if isinstance(result, Exception):
                logger.error(f"Error processing PDF {pdf_files[i]}: {result}")
            elif result:
                all_chunks.extend(result)
                processed_count += 1
                # Check if it was from cache
                if await self.is_pdf_cached(pdf_files[i]):
                    cached_count += 1

        logger.info(f"Processed {processed_count} PDFs ({cached_count} from cache, {processed_count - cached_count} newly parsed)")
        return all_chunks

    async def ingest_and_index(self, pdf_folder: str, index_path: str, meta_path: str):
        """
        Complete ingestion pipeline: process PDFs, chunk, embed, and save index.
        """
        pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))
        if not pdf_files:
            logger.warning(f"No PDF files found in '{pdf_folder}'.")
            return

        logger.info(f"Processing {len(pdf_files)} PDF files...")

        # Process all PDFs concurrently
        all_chunks = await self.process_pdf_batch(pdf_files)

        if all_chunks:
            logger.info(f"Generated {len(all_chunks)} chunks from {len(pdf_files)} PDFs")
            index, metadata = await self.build_faiss_index(all_chunks)
            await self.save_index(index, metadata, index_path, meta_path)
        else:
            logger.warning("No text was extracted from the PDFs. Index not built.")

    async def clear_cache(self):
        """Clear all cached text files."""
        try:
            import shutil
            if os.path.exists(self.cache_folder):
                shutil.rmtree(self.cache_folder)
                os.makedirs(self.cache_folder, exist_ok=True)
                logger.info("Cache cleared successfully")
        except Exception as e:
            logger.error(f"Error clearing cache: {e}")

    async def get_cache_stats(self) -> dict:
        """Get statistics about cached files."""
        try:
            cache_files = glob.glob(os.path.join(self.cache_folder, "*.txt"))
            total_size = sum(os.path.getsize(f) for f in cache_files)
            return {
                "cached_files": len(cache_files),
                "total_size_mb": total_size / (1024 * 1024),
                "cache_folder": self.cache_folder
            }
        except Exception as e:
            logger.error(f"Error getting cache stats: {e}")
    def __del__(self):
        """Cleanup executor on destruction."""
        if hasattr(self, 'executor'):
            self.executor.shutdown(wait=True)


# --- Async Chainlit Integration ---
# Example of how to structure the Chainlit part with async support

"""
import chainlit as cl

# Global variables
rag_system = None
index = None
metadata = None

@cl.on_chat_start
async def start():
    global rag_system, index, metadata

    try:
        rag_system = AsyncPDFRAG()
        index, metadata = await rag_system.load_index("my_document_index.faiss", "my_document_meta.pkl")
        await cl.Message(content="Hello! I'm ready to answer questions about your documents.").send()
    except FileNotFoundError:
        await cl.Message(content="The document index is not available. Please run the ingestion script.").send()
    except Exception as e:
        await cl.Message(content=f"Error loading index: {e}").send()

@cl.on_message
async def main(message: cl.Message):
    global rag_system, index, metadata

    if not index or not rag_system:
        await cl.Message(content="Sorry, I cannot answer questions without the document index.").send()
        return

    query = message.content

    try:
        # Retrieve relevant context from the FAISS index
        hits = await rag_system.query_index(query, index, metadata)
        context = "\n\n".join(hits)

        # Create a prompt for the RAG model
        prompt = (
            "You are an expert Q&A assistant who answers questions based on the provided context.\n"
            "Use ONLY the following context to answer the question. If the answer is not in the context, say you don't know.\n\n"
            f"Context:\n---\n{context}\n---\n\n"
            f"Question: {query}\n"
            "Answer:"
        )

        # Generate the answer asynchronously
        def _generate_response():
            return rag_system.parsing_model.generate_content(prompt)

        loop = asyncio.get_event_loop()
        response = await loop.run_in_executor(rag_system.executor, _generate_response)

        await cl.Message(content=response.text).send()

    except Exception as e:
        await cl.Message(content=f"Error processing query: {e}").send()
"""

# --- Example Usage ---
async def main():
    # Configuration
    PDF_FOLDER_PATH = '/content/pdfs_20'
    INDEX_FILE_PATH = 'my_document_index.faiss'
    METADATA_FILE_PATH = 'my_document_meta.pkl'
    CACHE_FOLDER = 'txt_cache'  # Custom cache folder

    # Create the PDF folder if it doesn't exist
    if not os.path.exists(PDF_FOLDER_PATH):
        os.makedirs(PDF_FOLDER_PATH)
        logger.info(f"Created folder '{PDF_FOLDER_PATH}'. Please add your PDF files there.")
        return

    # Initialize the RAG system with custom cache folder
    rag_system = AsyncPDFRAG(cache_folder=CACHE_FOLDER)

    try:
        # Show cache statistics
        cache_stats = await rag_system.get_cache_stats()
        logger.info(f"Cache stats: {cache_stats}")

        # Run the ingestion and indexing pipeline
        await rag_system.ingest_and_index(PDF_FOLDER_PATH, INDEX_FILE_PATH, METADATA_FILE_PATH)

        # Show updated cache statistics
        cache_stats = await rag_system.get_cache_stats()
        logger.info(f"Updated cache stats: {cache_stats}")

        # Example query after indexing
        if os.path.exists(INDEX_FILE_PATH):
            index, metadata = await rag_system.load_index(INDEX_FILE_PATH, METADATA_FILE_PATH)

            # Test query
            test_query = "What is the main topic of the documents?"
            results = await rag_system.query_index(test_query, index, metadata)

            logger.info(f"Query: {test_query}")
            logger.info(f"Results: {len(results)} chunks retrieved")

    except Exception as e:
        logger.error(f"Error in main pipeline: {e}")
    finally:
        # Cleanup is handled automatically by __del__
        pass

if __name__ == '__main__':
    # Check if we're in a notebook environment (Jupyter/Colab)
    try:
        # Try to get the current event loop
        loop = asyncio.get_running_loop()
        # If we get here, we're in a notebook with an existing event loop
        import nest_asyncio
        nest_asyncio.apply()  # Allow nested event loops
        asyncio.run(main())
    except RuntimeError:
        # No event loop running, we can use asyncio.run() normally
        asyncio.run(main())
    except ImportError:
        # nest_asyncio not available, use alternative approach
        try:
            # Get the current event loop
            loop = asyncio.get_event_loop()
            # Run the main function
            loop.run_until_complete(main())
        except RuntimeError:
            # Create a new event loop if none exists
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            loop.run_until_complete(main())
            loop.close()

In [ ]:
# Dependencies (install via pip)
# pip install google-generativeai langchain faiss-cpu pdfplumber numpy chainlit aiofiles

import os
import glob
import faiss
import pickle
import asyncio
import aiofiles
import google.generativeai as genai
import numpy as np
from langchain.text_splitter import RecursiveCharacterTextSplitter
from typing import List, Tuple, Optional
from concurrent.futures import ThreadPoolExecutor
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- Configuration ---
genai.configure(api_key=f"{GEMINI_API_KEY}")

PARSING_MODEL = "gemini-2.5-flash-latest"
EMBEDDING_MODEL = "text-embedding-004"

# Configuration for async processing
MAX_CONCURRENT_PDFS = 3  # Adjust based on API limits
MAX_CONCURRENT_CHUNKS = 50  # For embedding batches
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
CACHE_FOLDER = "txt_cache"  # Folder to store parsed text files

class AsyncPDFRAG:
    def __init__(self, max_concurrent_pdfs: int = MAX_CONCURRENT_PDFS,
                 max_concurrent_chunks: int = MAX_CONCURRENT_CHUNKS,
                 cache_folder: str = CACHE_FOLDER):
        self.max_concurrent_pdfs = max_concurrent_pdfs
        self.max_concurrent_chunks = max_concurrent_chunks
        self.cache_folder = cache_folder
        self.parsing_model = genai.GenerativeModel(PARSING_MODEL)
        self.executor = ThreadPoolExecutor(max_workers=max_concurrent_pdfs)

        # Create cache folder if it doesn't exist
        os.makedirs(self.cache_folder, exist_ok=True)

    def get_cache_path(self, pdf_path: str) -> str:
        """Generate cache file path for a given PDF."""
        pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
        return os.path.join(self.cache_folder, f"{pdf_name}.txt")

    async def is_pdf_cached(self, pdf_path: str) -> bool:
        """Check if PDF has already been parsed and cached."""
        cache_path = self.get_cache_path(pdf_path)
        if not os.path.exists(cache_path):
            return False

        # Check if cache is newer than the PDF file
        try:
            pdf_mtime = os.path.getmtime(pdf_path)
            cache_mtime = os.path.getmtime(cache_path)
            return cache_mtime > pdf_mtime
        except OSError:
            return False

    async def load_from_cache(self, pdf_path: str) -> Optional[str]:
        """Load parsed text from cache if available."""
        cache_path = self.get_cache_path(pdf_path)
        try:
            async with aiofiles.open(cache_path, 'r', encoding='utf-8') as f:
                content = await f.read()
                logger.info(f"Loaded cached text for {os.path.basename(pdf_path)}")
                return content
        except Exception as e:
            logger.error(f"Error loading cache for {pdf_path}: {e}")
            return None

    async def save_to_cache(self, pdf_path: str, content: str):
        """Save parsed text to cache."""
        cache_path = self.get_cache_path(pdf_path)
        try:
            async with aiofiles.open(cache_path, 'w', encoding='utf-8') as f:
                await f.write(content)
                logger.info(f"Cached parsed text for {os.path.basename(pdf_path)}")
        except Exception as e:
            logger.error(f"Error saving cache for {pdf_path}: {e}")

    async def parse_pdf_with_gemini(self, pdf_path: str) -> str:
        """
        Asynchronously parse a PDF file using Gemini 2.5 Pro with caching.
        """
        # Check if already cached
        if await self.is_pdf_cached(pdf_path):
            cached_content = await self.load_from_cache(pdf_path)
            if cached_content:
                return cached_content

        logger.info(f"Parsing {os.path.basename(pdf_path)} with Gemini 2.5 Pro...")

        def _upload_and_parse():
            try:
                uploaded_file = genai.upload_file(
                    path=pdf_path,
                    display_name=os.path.basename(pdf_path)
                )

                prompt = (
                    "You are a document intelligence assistant. Your task is to analyze the attached PDF file.\n"
                    "Please parse the document and generate a clean .txt output that includes the following:\n"
                    "- A clear document title.\n"
                    "- Key metadata such as author, publication date, and page count, if available.\n"
                    "- Structured headings and subheadings to preserve the document's layout.\n"
                    "- The full narrative text under each corresponding heading.\n"
                    "Ensure your response contains only the structured text output, ready for saving to a file."
                )

                response = self.parsing_model.generate_content([prompt, uploaded_file])
                return response.text
            except Exception as e:
                logger.error(f"Error parsing PDF {pdf_path}: {e}")
                return ""

        # Run the blocking operation in a thread pool
        loop = asyncio.get_event_loop()
        parsed_text = await loop.run_in_executor(self.executor, _upload_and_parse)

        # Cache the result if parsing was successful
        if parsed_text:
            await self.save_to_cache(pdf_path, parsed_text)

        return parsed_text

    def chunk_text(self, full_text: str) -> List[str]:
        """Split text into semantically coherent chunks."""
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", " ", ""],
            length_function=len
        )
        return splitter.split_text(full_text)

    async def embed_chunks_batch(self, chunks: List[str]) -> np.ndarray:
        """
        Embed chunks in batches asynchronously.
        """
        def _embed_batch(chunk_batch):
            try:
                result = genai.embed_content(
                    model=EMBEDDING_MODEL,
                    content=chunk_batch,
                    task_type="retrieval_document"
                )
                return result["embedding"]
            except Exception as e:
                logger.error(f"Error embedding batch: {e}")
                return []

        # Split chunks into batches to avoid API limits
        batch_size = min(self.max_concurrent_chunks, len(chunks))
        batches = [chunks[i:i + batch_size] for i in range(0, len(chunks), batch_size)]

        logger.info(f"Embedding {len(chunks)} chunks in {len(batches)} batches...")

        # Run batches concurrently
        loop = asyncio.get_event_loop()
        tasks = [loop.run_in_executor(self.executor, _embed_batch, batch) for batch in batches]

        embeddings_batches = await asyncio.gather(*tasks)

        # Flatten the results
        all_embeddings = []
        for batch_embeddings in embeddings_batches:
            if batch_embeddings:
                all_embeddings.extend(batch_embeddings)

        return np.array(all_embeddings, dtype="float32")

    async def build_faiss_index(self, chunks: List[str]) -> Tuple[faiss.Index, List[str]]:
        """Build FAISS index asynchronously."""
        if not chunks:
            raise ValueError("No chunks provided for indexing")

        embeddings_np = await self.embed_chunks_batch(chunks)

        if len(embeddings_np) == 0:
            raise ValueError("Failed to generate embeddings")

        # Build the FAISS index
        dim = embeddings_np.shape[1]
        index = faiss.IndexFlatL2(dim)
        index.add(embeddings_np)

        logger.info("FAISS index built successfully.")
        return index, chunks.copy()

    async def save_index(self, index: faiss.Index, metadata: List[str],
                        index_path: str, meta_path: str):
        """Save FAISS index and metadata asynchronously."""
        def _save_index():
            faiss.write_index(index, index_path)

        def _save_metadata():
            with open(meta_path, "wb") as f:
                pickle.dump(metadata, f)

        loop = asyncio.get_event_loop()
        await asyncio.gather(
            loop.run_in_executor(self.executor, _save_index),
            loop.run_in_executor(self.executor, _save_metadata)
        )

        logger.info(f"Index saved to {index_path} and metadata to {meta_path}")

    async def load_index(self, index_path: str, meta_path: str) -> Tuple[faiss.Index, List[str]]:
        """Load FAISS index and metadata asynchronously."""
        def _load_index():
            return faiss.read_index(index_path)

        def _load_metadata():
            with open(meta_path, "rb") as f:
                return pickle.load(f)

        loop = asyncio.get_event_loop()
        index, metadata = await asyncio.gather(
            loop.run_in_executor(self.executor, _load_index),
            loop.run_in_executor(self.executor, _load_metadata)
        )

        logger.info("Index and metadata loaded successfully.")
        return index, metadata

    async def query_index(self, query: str, index: faiss.Index,
                         metadata: List[str], top_k: int = 5) -> List[str]:
        """Query the index asynchronously."""
        def _embed_query():
            return genai.embed_content(
                model=EMBEDDING_MODEL,
                content=query,
                task_type="retrieval_query"
            )["embedding"]

        def _search_index(q_emb):
            distances, indices = index.search(np.array([q_emb], dtype="float32"), top_k)
            return [metadata[i] for i in indices[0]]

        loop = asyncio.get_event_loop()
        q_emb = await loop.run_in_executor(self.executor, _embed_query)
        results = await loop.run_in_executor(self.executor, _search_index, q_emb)

        return results

    def select_pdf_files(self, pdf_files: List[str], limit: Optional[int] = None,
                        offset: int = 0) -> List[str]:
        """
        Select a subset of PDF files based on offset and limit.

        Args:
            pdf_files: List of PDF file paths
            limit: Maximum number of files to process (None for all)
            offset: Number of files to skip from the beginning

        Returns:
            List of selected PDF file paths
        """
        total_files = len(pdf_files)

        if offset >= total_files:
            logger.warning(f"Offset {offset} is >= total files {total_files}. No files selected.")
            return []

        # Sort files to ensure consistent ordering
        sorted_files = sorted(pdf_files)

        # Apply offset
        selected_files = sorted_files[offset:]

        # Apply limit if specified
        if limit is not None:
            selected_files = selected_files[:limit]

        logger.info(f"Selected {len(selected_files)} files (offset: {offset}, limit: {limit}) from {total_files} total files")

        # Log the selected files for transparency
        for i, file in enumerate(selected_files):
            logger.info(f"  {i+1}. {os.path.basename(file)}")

        return selected_files

    async def process_pdf_batch(self, pdf_files: List[str]) -> List[str]:
        """Process multiple PDFs concurrently with caching support."""
        semaphore = asyncio.Semaphore(self.max_concurrent_pdfs)

        async def _process_single_pdf(pdf_file):
            async with semaphore:
                # Check cache first
                if await self.is_pdf_cached(pdf_file):
                    cached_content = await self.load_from_cache(pdf_file)
                    if cached_content:
                        return self.chunk_text(cached_content)

                # Parse if not cached
                structured_text = await self.parse_pdf_with_gemini(pdf_file)
                if structured_text:
                    return self.chunk_text(structured_text)
                return []

        # Process PDFs concurrently
        tasks = [_process_single_pdf(pdf_file) for pdf_file in pdf_files]
        results = await asyncio.gather(*tasks, return_exceptions=True)

        # Flatten results and handle exceptions
        all_chunks = []
        processed_count = 0
        cached_count = 0

        for i, result in enumerate(results):
            if isinstance(result, Exception):
                logger.error(f"Error processing PDF {pdf_files[i]}: {result}")
            elif result:
                all_chunks.extend(result)
                processed_count += 1
                # Check if it was from cache
                if await self.is_pdf_cached(pdf_files[i]):
                    cached_count += 1

        logger.info(f"Processed {processed_count} PDFs ({cached_count} from cache, {processed_count - cached_count} newly parsed)")
        return all_chunks

    async def ingest_and_index(self, pdf_folder: str, index_path: str, meta_path: str,
                              limit: Optional[int] = None, offset: int = 0):
        """
        Complete ingestion pipeline: process PDFs, chunk, embed, and save index.

        Args:
            pdf_folder: Path to folder containing PDF files
            index_path: Path to save the FAISS index
            meta_path: Path to save the metadata
            limit: Maximum number of PDFs to process (None for all)
            offset: Number of PDFs to skip from the beginning
        """
        all_pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))
        if not all_pdf_files:
            logger.warning(f"No PDF files found in '{pdf_folder}'.")
            return

        # Select subset of files based on offset and limit
        pdf_files = self.select_pdf_files(all_pdf_files, limit=limit, offset=offset)

        if not pdf_files:
            logger.warning("No PDF files selected for processing.")
            return

        logger.info(f"Processing {len(pdf_files)} PDF files (from {len(all_pdf_files)} total)...")

        # Process selected PDFs concurrently
        all_chunks = await self.process_pdf_batch(pdf_files)

        if all_chunks:
            logger.info(f"Generated {len(all_chunks)} chunks from {len(pdf_files)} PDFs")
            index, metadata = await self.build_faiss_index(all_chunks)
            await self.save_index(index, metadata, index_path, meta_path)
        else:
            logger.warning("No text was extracted from the PDFs. Index not built.")

    async def clear_cache(self):
        """Clear all cached text files."""
        try:
            import shutil
            if os.path.exists(self.cache_folder):
                shutil.rmtree(self.cache_folder)
                os.makedirs(self.cache_folder, exist_ok=True)
                logger.info("Cache cleared successfully")
        except Exception as e:
            logger.error(f"Error clearing cache: {e}")

    async def get_cache_stats(self) -> dict:
        """Get statistics about cached files."""
        try:
            cache_files = glob.glob(os.path.join(self.cache_folder, "*.txt"))
            total_size = sum(os.path.getsize(f) for f in cache_files)
            return {
                "cached_files": len(cache_files),
                "total_size_mb": total_size / (1024 * 1024),
                "cache_folder": self.cache_folder
            }
        except Exception as e:
            logger.error(f"Error getting cache stats: {e}")
            return {"error": str(e)}

    def __del__(self):
        """Cleanup executor on destruction."""
        if hasattr(self, 'executor'):
            self.executor.shutdown(wait=True)


# --- Async Chainlit Integration ---
# Example of how to structure the Chainlit part with async support

"""
import chainlit as cl

# Global variables
rag_system = None
index = None
metadata = None

@cl.on_chat_start
async def start():
    global rag_system, index, metadata

    try:
        rag_system = AsyncPDFRAG()
        index, metadata = await rag_system.load_index("my_document_index.faiss", "my_document_meta.pkl")
        await cl.Message(content="Hello! I'm ready to answer questions about your documents.").send()
    except FileNotFoundError:
        await cl.Message(content="The document index is not available. Please run the ingestion script.").send()
    except Exception as e:
        await cl.Message(content=f"Error loading index: {e}").send()

@cl.on_message
async def main(message: cl.Message):
    global rag_system, index, metadata

    if not index or not rag_system:
        await cl.Message(content="Sorry, I cannot answer questions without the document index.").send()
        return

    query = message.content

    try:
        # Retrieve relevant context from the FAISS index
        hits = await rag_system.query_index(query, index, metadata)
        context = "\n\n".join(hits)

        # Create a prompt for the RAG model
        prompt = (
            "You are an expert Q&A assistant who answers questions based on the provided context.\n"
            "Use ONLY the following context to answer the question. If the answer is not in the context, say you don't know.\n\n"
            f"Context:\n---\n{context}\n---\n\n"
            f"Question: {query}\n"
            "Answer:"
        )

        # Generate the answer asynchronously
        def _generate_response():
            return rag_system.parsing_model.generate_content(prompt)

        loop = asyncio.get_event_loop()
        response = await loop.run_in_executor(rag_system.executor, _generate_response)

        await cl.Message(content=response.text).send()

    except Exception as e:
        await cl.Message(content=f"Error processing query: {e}").send()
"""

# --- Example Usage ---
async def main():
    # Configuration
    PDF_FOLDER_PATH = '/content/pdfs_20'
    INDEX_FILE_PATH = 'my_document_index.faiss'
    METADATA_FILE_PATH = 'my_document_meta.pkl'
    CACHE_FOLDER = 'txt_cache'  # Custom cache folder

    # Test configuration - process only 3 documents starting from offset 0
    TEST_LIMIT = 2
    TEST_OFFSET = 0

    # Create the PDF folder if it doesn't exist
    if not os.path.exists(PDF_FOLDER_PATH):
        os.makedirs(PDF_FOLDER_PATH)
        logger.info(f"Created folder '{PDF_FOLDER_PATH}'. Please add your PDF files there.")
        return

    # Initialize the RAG system with custom cache folder
    rag_system = AsyncPDFRAG(cache_folder=CACHE_FOLDER)

    try:
        # Show cache statistics
        cache_stats = await rag_system.get_cache_stats()
        logger.info(f"Cache stats: {cache_stats}")

        # Run the ingestion and indexing pipeline with offset and limit
        logger.info(f"Testing with {TEST_LIMIT} documents starting from offset {TEST_OFFSET}")
        await rag_system.ingest_and_index(
            PDF_FOLDER_PATH,
            INDEX_FILE_PATH,
            METADATA_FILE_PATH,
            limit=TEST_LIMIT,
            offset=TEST_OFFSET
        )

        # Show updated cache statistics
        cache_stats = await rag_system.get_cache_stats()
        logger.info(f"Updated cache stats: {cache_stats}")

        # Example query after indexing
        if os.path.exists(INDEX_FILE_PATH):
            index, metadata = await rag_system.load_index(INDEX_FILE_PATH, METADATA_FILE_PATH)

            # Test query
            test_query = "What is the main topic of the documents?"
            results = await rag_system.query_index(test_query, index, metadata)

            logger.info(f"Query: {test_query}")
            logger.info(f"Results: {len(results)} chunks retrieved")

            # Show sample results
            for i, result in enumerate(results[:2]):  # Show first 2 results
                logger.info(f"Result {i+1}: {result[:200]}...")

    except Exception as e:
        logger.error(f"Error in main pipeline: {e}")
    finally:
        # Cleanup is handled automatically by __del__
        pass

# --- Alternative Example: Processing different subsets ---
async def example_different_subsets():
    """Example showing how to process different subsets of documents."""
    rag_system = AsyncPDFRAG()

    # Example 1: Process first 3 documents
    logger.info("=== Processing first 3 documents ===")
    await rag_system.ingest_and_index(
        '/content/pdfs_20',
        'index_first_3.faiss',
        'meta_first_3.pkl',
        limit=3,
        offset=0
    )



if __name__ == '__main__':
    # Check if we're in a notebook environment (Jupyter/Colab)
    try:
        # Try to get the current event loop
        loop = asyncio.get_running_loop()
        # If we get here, we're in a notebook with an existing event loop
        import nest_asyncio
        nest_asyncio.apply()  # Allow nested event loops
        asyncio.run(main())
    except RuntimeError:
        # No event loop running, we can use asyncio.run() normally
        asyncio.run(main())
    except ImportError:
        # nest_asyncio not available, use alternative approach
        try:
            # Get the current event loop
            loop = asyncio.get_event_loop()
            # Run the main function
            loop.run_until_complete(main())
        except RuntimeError:
            # Create a new event loop if none exists
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            loop.run_until_complete(main())
            loop.close()

In [8]:
 !pip install google-generativeai langchain faiss-cpu pdfplumber numpy chainlit aiofiles pypdf2 pymupdf

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Dependencies (install via pip)
# pip install google-generativeai langchain faiss-cpu pdfplumber numpy chainlit aiofiles pypdf2 pymupdf

import os
import glob
import faiss
import pickle
import asyncio
import aiofiles
import google.generativeai as genai
import numpy as np
from langchain.text_splitter import RecursiveCharacterTextSplitter
from typing import List, Tuple, Optional, Dict, Any
from concurrent.futures import ThreadPoolExecutor
import logging
from enum import Enum
from dataclasses import dataclass

# PDF processing libraries
import pdfplumber
import PyPDF2
import fitz  # PyMuPDF

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- Configuration ---
genai.configure(api_key=f"{GEMINI_API_KEY}")

PARSING_MODEL = "gemini-2.5-flash-latest"
EMBEDDING_MODEL = "text-embedding-004"

# Configuration for async processing
MAX_CONCURRENT_PDFS = 3  # Adjust based on API limits
MAX_CONCURRENT_CHUNKS = 50  # For embedding batches
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
CACHE_FOLDER = "txt_cache"  # Folder to store parsed text files

class PDFParsingMethod(Enum):
    """Enumeration of available PDF parsing methods."""
    GEMINI = "gemini"
    PDFPLUMBER = "pdfplumber"
    PYPDF2 = "pypdf2"
    PYMUPDF = "pymupdf"
    AUTO = "auto"  # Try multiple methods and pick the best result

@dataclass
class PDFParsingConfig:
    """Configuration for PDF parsing methods."""
    method: PDFParsingMethod = PDFParsingMethod.PYPDF2
    gemini_fallback: bool = True  # Use Gemini as fallback if traditional methods fail
    extract_images: bool = False  # Extract image descriptions (only with Gemini)
    extract_tables: bool = True   # Extract table content
    preserve_formatting: bool = True  # Preserve text formatting
    min_text_length: int = 100    # Minimum text length to consider parsing successful

class AsyncPDFRAG:
    def __init__(self, max_concurrent_pdfs: int = MAX_CONCURRENT_PDFS,
                 max_concurrent_chunks: int = MAX_CONCURRENT_CHUNKS,
                 cache_folder: str = CACHE_FOLDER,
                 parsing_config: PDFParsingConfig = None):
        self.max_concurrent_pdfs = max_concurrent_pdfs
        self.max_concurrent_chunks = max_concurrent_chunks
        self.cache_folder = cache_folder
        self.parsing_config = parsing_config or PDFParsingConfig()
        self.parsing_model = genai.GenerativeModel(PARSING_MODEL)
        self.executor = ThreadPoolExecutor(max_workers=max_concurrent_pdfs)

        # Create cache folder if it doesn't exist
        os.makedirs(self.cache_folder, exist_ok=True)

    def get_cache_path(self, pdf_path: str, method: str = None) -> str:
        """Generate cache file path for a given PDF and method."""
        pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
        method_suffix = f"_{method}" if method else ""
        return os.path.join(self.cache_folder, f"{pdf_name}{method_suffix}.txt")

    async def is_pdf_cached(self, pdf_path: str, method: str = None) -> bool:
        """Check if PDF has already been parsed and cached."""
        cache_path = self.get_cache_path(pdf_path, method)
        if not os.path.exists(cache_path):
            return False

        # Check if cache is newer than the PDF file
        try:
            pdf_mtime = os.path.getmtime(pdf_path)
            cache_mtime = os.path.getmtime(cache_path)
            return cache_mtime > pdf_mtime
        except OSError:
            return False

    async def load_from_cache(self, pdf_path: str, method: str = None) -> Optional[str]:
        """Load parsed text from cache if available."""
        cache_path = self.get_cache_path(pdf_path, method)
        try:
            async with aiofiles.open(cache_path, 'r', encoding='utf-8') as f:
                content = await f.read()
                method_info = f" ({method})" if method else ""
                logger.info(f"Loaded cached text for {os.path.basename(pdf_path)}{method_info}")
                return content
        except Exception as e:
            logger.error(f"Error loading cache for {pdf_path}: {e}")
            return None

    async def save_to_cache(self, pdf_path: str, content: str, method: str = None):
        """Save parsed text to cache."""
        cache_path = self.get_cache_path(pdf_path, method)
        try:
            async with aiofiles.open(cache_path, 'w', encoding='utf-8') as f:
                await f.write(content)
                method_info = f" ({method})" if method else ""
                logger.info(f"Cached parsed text for {os.path.basename(pdf_path)}{method_info}")
        except Exception as e:
            logger.error(f"Error saving cache for {pdf_path}: {e}")

    def parse_pdf_with_pdfplumber(self, pdf_path: str) -> str:
        """Parse PDF using pdfplumber library."""
        try:
            text_content = []

            with pdfplumber.open(pdf_path) as pdf:
                # Extract metadata
                metadata = pdf.metadata or {}
                if metadata:
                    text_content.append("=== DOCUMENT METADATA ===")
                    for key, value in metadata.items():
                        if value:
                            text_content.append(f"{key}: {value}")
                    text_content.append("")

                # Extract text from each page
                for page_num, page in enumerate(pdf.pages, 1):
                    page_text = page.extract_text()
                    if page_text and page_text.strip():
                        text_content.append(f"=== PAGE {page_num} ===")
                        text_content.append(page_text.strip())
                        text_content.append("")

                    # Extract tables if configured
                    if self.parsing_config.extract_tables:
                        tables = page.extract_tables()
                        for table_num, table in enumerate(tables, 1):
                            text_content.append(f"--- Table {table_num} on Page {page_num} ---")
                            for row in table:
                                if row:
                                    text_content.append(" | ".join(str(cell) if cell else "" for cell in row))
                            text_content.append("")

            return "\n".join(text_content)

        except Exception as e:
            logger.error(f"Error parsing PDF with pdfplumber {pdf_path}: {e}")
            return ""

    def parse_pdf_with_pypdf2(self, pdf_path: str) -> str:
        """Parse PDF using PyPDF2 library."""
        try:
            text_content = []

            with open(pdf_path, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)

                # Extract metadata
                metadata = pdf_reader.metadata
                if metadata:
                    text_content.append("=== DOCUMENT METADATA ===")
                    for key, value in metadata.items():
                        if value:
                            text_content.append(f"{key}: {value}")
                    text_content.append("")

                # Extract text from each page
                for page_num, page in enumerate(pdf_reader.pages, 1):
                    page_text = page.extract_text()
                    if page_text and page_text.strip():
                        text_content.append(f"=== PAGE {page_num} ===")
                        text_content.append(page_text.strip())
                        text_content.append("")

            return "\n".join(text_content)

        except Exception as e:
            logger.error(f"Error parsing PDF with PyPDF2 {pdf_path}: {e}")
            return ""

    def parse_pdf_with_pymupdf(self, pdf_path: str) -> str:
        """Parse PDF using PyMuPDF library."""
        try:
            text_content = []

            doc = fitz.open(pdf_path)

            # Extract metadata
            metadata = doc.metadata
            if metadata:
                text_content.append("=== DOCUMENT METADATA ===")
                for key, value in metadata.items():
                    if value:
                        text_content.append(f"{key}: {value}")
                text_content.append("")

            # Extract text from each page
            for page_num in range(len(doc)):
                page = doc[page_num]
                page_text = page.get_text()

                if page_text and page_text.strip():
                    text_content.append(f"=== PAGE {page_num + 1} ===")
                    text_content.append(page_text.strip())
                    text_content.append("")

                # Extract tables if configured
                if self.parsing_config.extract_tables:
                    tables = page.find_tables()
                    for table_num, table in enumerate(tables, 1):
                        text_content.append(f"--- Table {table_num} on Page {page_num + 1} ---")
                        table_data = table.extract()
                        for row in table_data:
                            if row:
                                text_content.append(" | ".join(str(cell) if cell else "" for cell in row))
                        text_content.append("")

            doc.close()
            return "\n".join(text_content)

        except Exception as e:
            logger.error(f"Error parsing PDF with PyMuPDF {pdf_path}: {e}")
            return ""

    async def parse_pdf_with_gemini(self, pdf_path: str) -> str:
        """Parse PDF using Gemini AI with enhanced prompt."""
        logger.info(f"Parsing {os.path.basename(pdf_path)} with Gemini AI...")

        def _upload_and_parse():
            try:
                uploaded_file = genai.upload_file(
                    path=pdf_path,
                    display_name=os.path.basename(pdf_path)
                )

                # Enhanced prompt based on configuration
                prompt_parts = [
                    "You are an advanced document intelligence assistant. Analyze the attached PDF file and generate a comprehensive .txt output."
                ]

                if self.parsing_config.extract_images:
                    prompt_parts.append("- Describe any images, charts, graphs, or diagrams in detail.")

                if self.parsing_config.extract_tables:
                    prompt_parts.append("- Extract and format all tables with clear structure.")

                if self.parsing_config.preserve_formatting:
                    prompt_parts.append("- Preserve the document's hierarchical structure with clear headings and subheadings.")

                prompt_parts.extend([
                    "- Include document metadata (title, author, date, etc.) if available.",
                    "- Maintain the logical flow and organization of the content.",
                    "- Use markdown-style formatting for better readability.",
                    "- Ensure the output is clean and ready for text processing.",
                    "",
                    "Focus on extracting meaningful content while preserving the document's structure and context."
                ])

                prompt = "\n".join(prompt_parts)

                response = self.parsing_model.generate_content([prompt, uploaded_file])
                return response.text
            except Exception as e:
                logger.error(f"Error parsing PDF with Gemini {pdf_path}: {e}")
                return ""

        # Run the blocking operation in a thread pool
        loop = asyncio.get_event_loop()
        parsed_text = await loop.run_in_executor(self.executor, _upload_and_parse)

        return parsed_text

    def evaluate_parsing_quality(self, text: str, pdf_path: str) -> Dict[str, Any]:
        """Evaluate the quality of parsed text."""
        if not text:
            return {"score": 0, "reason": "No text extracted"}

        # Basic quality metrics
        text_length = len(text.strip())
        word_count = len(text.split())
        line_count = len(text.split('\n'))

        # Quality indicators
        has_structure = any(marker in text for marker in ['===', '---', '#', '##'])
        has_metadata = 'METADATA' in text or 'metadata' in text.lower()

        # Calculate score
        score = 0
        reasons = []

        if text_length >= self.parsing_config.min_text_length:
            score += 40
        else:
            reasons.append(f"Text too short ({text_length} chars)")

        if word_count > 50:
            score += 20

        if has_structure:
            score += 20

        if has_metadata:
            score += 10

        if line_count > 10:
            score += 10

        return {
            "score": score,
            "text_length": text_length,
            "word_count": word_count,
            "line_count": line_count,
            "has_structure": has_structure,
            "has_metadata": has_metadata,
            "reasons": reasons
        }

    async def parse_pdf_auto(self, pdf_path: str) -> Tuple[str, str]:
        """
        Automatically try multiple parsing methods and return the best result.
        Returns (best_text, method_used)
        """
        methods_to_try = [
            (PDFParsingMethod.PDFPLUMBER, self.parse_pdf_with_pdfplumber),
            (PDFParsingMethod.PYMUPDF, self.parse_pdf_with_pymupdf),
            (PDFParsingMethod.PYPDF2, self.parse_pdf_with_pypdf2),
        ]

        best_result = ("", "none")
        best_score = 0

        # Try traditional methods first
        for method, parser_func in methods_to_try:
            try:
                loop = asyncio.get_event_loop()
                text = await loop.run_in_executor(self.executor, parser_func, pdf_path)

                if text:
                    quality = self.evaluate_parsing_quality(text, pdf_path)
                    logger.info(f"{method.value} parsing score: {quality['score']}")

                    if quality['score'] > best_score:
                        best_score = quality['score']
                        best_result = (text, method.value)

            except Exception as e:
                logger.error(f"Error with {method.value} parsing: {e}")
                continue

        # Use Gemini as fallback if traditional methods failed or score is low
        if (best_score < 60 and self.parsing_config.gemini_fallback) or best_score == 0:
            logger.info("Using Gemini AI as fallback for better parsing quality")
            gemini_text = await self.parse_pdf_with_gemini(pdf_path)
            if gemini_text:
                gemini_quality = self.evaluate_parsing_quality(gemini_text, pdf_path)
                logger.info(f"Gemini parsing score: {gemini_quality['score']}")

                if gemini_quality['score'] > best_score:
                    best_result = (gemini_text, PDFParsingMethod.GEMINI.value)

        return best_result

    async def parse_pdf_with_method(self, pdf_path: str, method: PDFParsingMethod = None) -> str:
        """
        Parse PDF using specified method or auto-detection.
        """
        method = method or self.parsing_config.method
        method_str = method.value if method != PDFParsingMethod.AUTO else "auto"

        # Check cache first
        if await self.is_pdf_cached(pdf_path, method_str):
            cached_content = await self.load_from_cache(pdf_path, method_str)
            if cached_content:
                return cached_content

        logger.info(f"Parsing {os.path.basename(pdf_path)} with method: {method_str}")

        # Parse based on method
        if method == PDFParsingMethod.AUTO:
            parsed_text, actual_method = await self.parse_pdf_auto(pdf_path)
            method_str = actual_method
        elif method == PDFParsingMethod.GEMINI:
            parsed_text = await self.parse_pdf_with_gemini(pdf_path)
        elif method == PDFParsingMethod.PDFPLUMBER:
            loop = asyncio.get_event_loop()
            parsed_text = await loop.run_in_executor(self.executor, self.parse_pdf_with_pdfplumber, pdf_path)
        elif method == PDFParsingMethod.PYPDF2:
            loop = asyncio.get_event_loop()
            parsed_text = await loop.run_in_executor(self.executor, self.parse_pdf_with_pypdf2, pdf_path)
        elif method == PDFParsingMethod.PYMUPDF:
            loop = asyncio.get_event_loop()
            parsed_text = await loop.run_in_executor(self.executor, self.parse_pdf_with_pymupdf, pdf_path)
        else:
            raise ValueError(f"Unknown parsing method: {method}")

        # Cache the result if parsing was successful
        if parsed_text:
            await self.save_to_cache(pdf_path, parsed_text, method_str)

        return parsed_text

    def chunk_text(self, full_text: str) -> List[str]:
        """Split text into semantically coherent chunks."""
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", " ", ""],
            length_function=len
        )
        return splitter.split_text(full_text)

    async def embed_chunks_batch(self, chunks: List[str]) -> np.ndarray:
        """
        Embed chunks in batches asynchronously.
        """
        def _embed_batch(chunk_batch):
            try:
                result = genai.embed_content(
                    model=EMBEDDING_MODEL,
                    content=chunk_batch,
                    task_type="retrieval_document"
                )
                return result["embedding"]
            except Exception as e:
                logger.error(f"Error embedding batch: {e}")
                return []

        # Split chunks into batches to avoid API limits
        batch_size = min(self.max_concurrent_chunks, len(chunks))
        batches = [chunks[i:i + batch_size] for i in range(0, len(chunks), batch_size)]

        logger.info(f"Embedding {len(chunks)} chunks in {len(batches)} batches...")

        # Run batches concurrently
        loop = asyncio.get_event_loop()
        tasks = [loop.run_in_executor(self.executor, _embed_batch, batch) for batch in batches]

        embeddings_batches = await asyncio.gather(*tasks)

        # Flatten the results
        all_embeddings = []
        for batch_embeddings in embeddings_batches:
            if batch_embeddings:
                all_embeddings.extend(batch_embeddings)

        return np.array(all_embeddings, dtype="float32")

    async def build_faiss_index(self, chunks: List[str]) -> Tuple[faiss.Index, List[str]]:
        """Build FAISS index asynchronously."""
        if not chunks:
            raise ValueError("No chunks provided for indexing")

        embeddings_np = await self.embed_chunks_batch(chunks)

        if len(embeddings_np) == 0:
            raise ValueError("Failed to generate embeddings")

        # Build the FAISS index
        dim = embeddings_np.shape[1]
        index = faiss.IndexFlatL2(dim)
        index.add(embeddings_np)

        logger.info("FAISS index built successfully.")
        return index, chunks.copy()

    async def save_index(self, index: faiss.Index, metadata: List[str],
                        index_path: str, meta_path: str):
        """Save FAISS index and metadata asynchronously."""
        def _save_index():
            faiss.write_index(index, index_path)

        def _save_metadata():
            with open(meta_path, "wb") as f:
                pickle.dump(metadata, f)

        loop = asyncio.get_event_loop()
        await asyncio.gather(
            loop.run_in_executor(self.executor, _save_index),
            loop.run_in_executor(self.executor, _save_metadata)
        )

        logger.info(f"Index saved to {index_path} and metadata to {meta_path}")

    async def load_index(self, index_path: str, meta_path: str) -> Tuple[faiss.Index, List[str]]:
        """Load FAISS index and metadata asynchronously."""
        def _load_index():
            return faiss.read_index(index_path)

        def _load_metadata():
            with open(meta_path, "rb") as f:
                return pickle.load(f)

        loop = asyncio.get_event_loop()
        index, metadata = await asyncio.gather(
            loop.run_in_executor(self.executor, _load_index),
            loop.run_in_executor(self.executor, _load_metadata)
        )

        logger.info("Index and metadata loaded successfully.")
        return index, metadata

    async def query_index(self, query: str, index: faiss.Index,
                         metadata: List[str], top_k: int = 5) -> List[str]:
        """Query the index asynchronously."""
        def _embed_query():
            return genai.embed_content(
                model=EMBEDDING_MODEL,
                content=query,
                task_type="retrieval_query"
            )["embedding"]

        def _search_index(q_emb):
            distances, indices = index.search(np.array([q_emb], dtype="float32"), top_k)
            return [metadata[i] for i in indices[0]]

        loop = asyncio.get_event_loop()
        q_emb = await loop.run_in_executor(self.executor, _embed_query)
        results = await loop.run_in_executor(self.executor, _search_index, q_emb)

        return results

    def select_pdf_files(self, pdf_files: List[str], limit: Optional[int] = None,
                        offset: int = 0) -> List[str]:
        """
        Select a subset of PDF files based on offset and limit.
        """
        total_files = len(pdf_files)

        if offset >= total_files:
            logger.warning(f"Offset {offset} is >= total files {total_files}. No files selected.")
            return []

        # Sort files to ensure consistent ordering
        sorted_files = sorted(pdf_files)

        # Apply offset
        selected_files = sorted_files[offset:]

        # Apply limit if specified
        if limit is not None:
            selected_files = selected_files[:limit]

        logger.info(f"Selected {len(selected_files)} files (offset: {offset}, limit: {limit}) from {total_files} total files")

        # Log the selected files for transparency
        for i, file in enumerate(selected_files):
            logger.info(f"  {i+1}. {os.path.basename(file)}")

        return selected_files

    async def process_pdf_batch(self, pdf_files: List[str]) -> List[str]:
        """Process multiple PDFs concurrently with flexible parsing methods."""
        semaphore = asyncio.Semaphore(self.max_concurrent_pdfs)

        async def _process_single_pdf(pdf_file):
            async with semaphore:
                structured_text = await self.parse_pdf_with_method(pdf_file)
                if structured_text:
                    return self.chunk_text(structured_text)
                return []

        # Process PDFs concurrently
        tasks = [_process_single_pdf(pdf_file) for pdf_file in pdf_files]
        results = await asyncio.gather(*tasks, return_exceptions=True)

        # Flatten results and handle exceptions
        all_chunks = []
        processed_count = 0

        for i, result in enumerate(results):
            if isinstance(result, Exception):
                logger.error(f"Error processing PDF {pdf_files[i]}: {result}")
            elif result:
                all_chunks.extend(result)
                processed_count += 1

        logger.info(f"Successfully processed {processed_count} PDFs")
        return all_chunks

    async def ingest_and_index(self, pdf_folder: str, index_path: str, meta_path: str,
                              limit: Optional[int] = None, offset: int = 0):
        """
        Complete ingestion pipeline with flexible parsing methods.
        """
        all_pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))
        if not all_pdf_files:
            logger.warning(f"No PDF files found in '{pdf_folder}'.")
            return

        # Select subset of files based on offset and limit
        pdf_files = self.select_pdf_files(all_pdf_files, limit=limit, offset=offset)

        if not pdf_files:
            logger.warning("No PDF files selected for processing.")
            return

        logger.info(f"Processing {len(pdf_files)} PDF files with method: {self.parsing_config.method.value}")

        # Process selected PDFs concurrently
        all_chunks = await self.process_pdf_batch(pdf_files)

        if all_chunks:
            logger.info(f"Generated {len(all_chunks)} chunks from {len(pdf_files)} PDFs")
            index, metadata = await self.build_faiss_index(all_chunks)
            await self.save_index(index, metadata, index_path, meta_path)
        else:
            logger.warning("No text was extracted from the PDFs. Index not built.")

    async def clear_cache(self):
        """Clear all cached text files."""
        try:
            import shutil
            if os.path.exists(self.cache_folder):
                shutil.rmtree(self.cache_folder)
                os.makedirs(self.cache_folder, exist_ok=True)
                logger.info("Cache cleared successfully")
        except Exception as e:
            logger.error(f"Error clearing cache: {e}")

    async def get_cache_stats(self) -> dict:
        """Get statistics about cached files."""
        try:
            cache_files = glob.glob(os.path.join(self.cache_folder, "*.txt"))
            total_size = sum(os.path.getsize(f) for f in cache_files)

            # Group by method
            method_stats = {}
            for file in cache_files:
                filename = os.path.basename(file)
                if '_' in filename:
                    method = filename.split('_')[-1].replace('.txt', '')
                    method_stats[method] = method_stats.get(method, 0) + 1

            return {
                "cached_files": len(cache_files),
                "total_size_mb": total_size / (1024 * 1024),
                "cache_folder": self.cache_folder,
                "methods": method_stats
            }
        except Exception as e:
            logger.error(f"Error getting cache stats: {e}")
            return {"error": str(e)}

    def __del__(self):
        """Cleanup executor on destruction."""
        if hasattr(self, 'executor'):
            self.executor.shutdown(wait=True)


# --- Example Usage ---
async def main():
    # Configuration
    PDF_FOLDER_PATH = '/pdfs_20_2'
    INDEX_FILE_PATH = 'doc_index.faiss'
    METADATA_FILE_PATH = 'meta_data_index.pkl'
    CACHE_FOLDER = 'txt_cache_2'

    # Test configuration
    TEST_LIMIT = 1
    TEST_OFFSET = 0

    # Configure parsing method
    parsing_config = PDFParsingConfig(
        method=PDFParsingMethod.PYPDF2,  # Try multiple methods automatically
        gemini_fallback=False,          # Use Gemini if traditional methods fail
        extract_images=False,           # Extract image descriptions with Gemini
        extract_tables=True,           # Extract table content
        preserve_formatting=True,      # Preserve document structure
        min_text_length=100           # Minimum text length for success
    )

    # Create the PDF folder if it doesn't exist


    # Initialize the RAG system
    rag_system = AsyncPDFRAG(
        cache_folder=CACHE_FOLDER,
        parsing_config=parsing_config
    )

    try:
        # Show cache statistics
        cache_stats = await rag_system.get_cache_stats()
        logger.info(f"Cache stats: {cache_stats}")

        # Run the ingestion and indexing pipeline
        logger.info(f"Testing with {TEST_LIMIT} documents starting from offset {TEST_OFFSET}")
        logger.info(f"Using parsing method: {parsing_config.method.value}")

        await rag_system.ingest_and_index(
            PDF_FOLDER_PATH,
            INDEX_FILE_PATH,
            METADATA_FILE_PATH,
            limit=TEST_LIMIT,
            offset=TEST_OFFSET
        )

        # Show updated cache statistics
        cache_stats = await rag_system.get_cache_stats()
        logger.info(f"Updated cache stats: {cache_stats}")

        # Example query after indexing
        if os.path.exists(INDEX_FILE_PATH):
            index, metadata = await rag_system.load_index(INDEX_FILE_PATH, METADATA_FILE_PATH)

            # Test query
            test_query = "What is the main topic of the documents?"
            results = await rag_system.query_index(test_query, index, metadata)

            logger.info(f"Query: {test_query}")
            logger.info(f"Results: {len(results)} chunks retrieved")

            # Show sample results
            for i, result in enumerate(results[:2]):
                logger.info(f"Result {i+1}: {result[:200]}...")

    except Exception as e:
        logger.error(f"Error in main pipeline: {e}")

# --- Example with different parsing methods ---
async def example_different_methods():
    """Example showing different parsing methods."""

    # Example 1: Pure traditional PDF parsing (no Gemini)
    config1 = PDFParsingConfig(
        method=PDFParsingMethod.PDFPLUMBER,
        gemini_fallback=False,
        extract_tables=True,
        preserve_formatting=True
    )

    # Example 2: PyMuPDF with Gemini fallback
    config2 = PDFParsingConfig(
        method=PDFParsingMethod.PYMUPDF,
        gemini_fallback=True,
        extract_tables=True,
        preserve_formatting=True
    )

    # Example 3: Gemini-only parsing with image extraction
    config3 = PDFParsingConfig(
        method=PDFParsingMethod.GEMINI,
        extract_images=True,
        extract_tables=True,
        preserve_formatting=True
    )

    # Example 4: Auto method (tries all and picks best)
    config4 = PDFParsingConfig(
        method=PDFParsingMethod.AUTO,
        gemini_fallback=True,
        extract_images=True,
        extract_tables=True,
        preserve_formatting=True
    )

    configs = [
        ("PDFPlumber Only", config1),
        ("PyMuPDF + Gemini Fallback", config2),
        ("Gemini Only", config3),
        ("Auto Selection", config4)
    ]

    for name, config in configs:
        logger.info(f"\n=== Testing {name} ===")

        rag_system = AsyncPDFRAG(
            cache_folder=f'cache_{name.lower().replace(" ", "_")}',
            parsing_config=config
        )

        try:
            await rag_system.ingest_and_index(
                '/content/pdfs_20',
                f'index_{name.lower().replace(" ", "_")}.faiss',
                f'meta_{name.lower().replace(" ", "_")}.pkl',
                limit=2,  # Test with 2 documents
                offset=0
            )

            # Show cache stats for this method
            cache_stats = await rag_system.get_cache_stats()
            logger.info(f"Cache stats for {name}: {cache_stats}")

        except Exception as e:
            logger.error(f"Error with {name}: {e}")

# --- Utility functions ---
async def compare_parsing_methods(pdf_path: str):
    """Compare different parsing methods on a single PDF."""

    methods = [
        (PDFParsingMethod.PDFPLUMBER, "PDFPlumber"),
        (PDFParsingMethod.PYPDF2, "PyPDF2"),
        (PDFParsingMethod.PYMUPDF, "PyMuPDF"),
        (PDFParsingMethod.GEMINI, "Gemini AI")
    ]

    results = {}

    for method, name in methods:
        config = PDFParsingConfig(method=method)
        rag_system = AsyncPDFRAG(parsing_config=config)

        try:
            logger.info(f"\n=== Testing {name} on {os.path.basename(pdf_path)} ===")
            start_time = asyncio.get_event_loop().time()

            text = await rag_system.parse_pdf_with_method(pdf_path, method)

            end_time = asyncio.get_event_loop().time()
            processing_time = end_time - start_time

            quality = rag_system.evaluate_parsing_quality(text, pdf_path)

            results[name] = {
                "text_length": len(text),
                "word_count": len(text.split()),
                "processing_time": processing_time,
                "quality_score": quality["score"],
                "quality_details": quality
            }

            logger.info(f"{name} - Length: {len(text)}, Score: {quality['score']}, Time: {processing_time:.2f}s")

        except Exception as e:
            logger.error(f"Error with {name}: {e}")
            results[name] = {"error": str(e)}

    return results

async def batch_process_with_different_methods():
    """Process different batches of documents with different methods."""

    # Method configurations
    method_configs = {
        "fast_traditional": PDFParsingConfig(
            method=PDFParsingMethod.PDFPLUMBER,
            gemini_fallback=False,
            extract_tables=False,
            preserve_formatting=False
        ),
        "comprehensive_traditional": PDFParsingConfig(
            method=PDFParsingMethod.PYMUPDF,
            gemini_fallback=False,
            extract_tables=True,
            preserve_formatting=True
        ),
        "ai_enhanced": PDFParsingConfig(
            method=PDFParsingMethod.AUTO,
            gemini_fallback=True,
            extract_images=True,
            extract_tables=True,
            preserve_formatting=True
        ),
        "gemini_only": PDFParsingConfig(
            method=PDFParsingMethod.GEMINI,
            extract_images=True,
            extract_tables=True,
            preserve_formatting=True
        )
    }

    # Process different subsets with different methods
    for method_name, config in method_configs.items():
        logger.info(f"\n=== Processing with {method_name} ===")

        rag_system = AsyncPDFRAG(
            cache_folder=f'cache_{method_name}',
            parsing_config=config
        )

        try:
            await rag_system.ingest_and_index(
                '/content/pdfs_20',
                f'index_{method_name}.faiss',
                f'meta_{method_name}.pkl',
                limit=3,
                offset=0
            )

            # Show performance stats
            cache_stats = await rag_system.get_cache_stats()
            logger.info(f"Performance stats for {method_name}: {cache_stats}")

        except Exception as e:
            logger.error(f"Error with {method_name}: {e}")

# --- Advanced Configuration Examples ---
def create_speed_optimized_config():
    """Configuration optimized for speed."""
    return PDFParsingConfig(
        method=PDFParsingMethod.PYPDF2,
        gemini_fallback=False,
        extract_images=False,
        extract_tables=False,
        preserve_formatting=False,
        min_text_length=50
    )

def create_quality_optimized_config():
    """Configuration optimized for quality."""
    return PDFParsingConfig(
        method=PDFParsingMethod.AUTO,
        gemini_fallback=True,
        extract_images=True,
        extract_tables=True,
        preserve_formatting=True,
        min_text_length=200
    )

def create_balanced_config():
    """Balanced configuration for speed and quality."""
    return PDFParsingConfig(
        method=PDFParsingMethod.PDFPLUMBER,
        gemini_fallback=True,
        extract_images=False,
        extract_tables=True,
        preserve_formatting=True,
        min_text_length=100
    )

# --- Performance Testing ---
async def performance_benchmark():
    """Benchmark different parsing methods."""

    test_configs = {
        "Speed": create_speed_optimized_config(),
        "Quality": create_quality_optimized_config(),
        "Balanced": create_balanced_config()
    }

    pdf_folder = '/content/pdfs_20'

    for config_name, config in test_configs.items():
        logger.info(f"\n=== Benchmarking {config_name} Configuration ===")

        rag_system = AsyncPDFRAG(
            cache_folder=f'benchmark_{config_name.lower()}',
            parsing_config=config
        )

        start_time = asyncio.get_event_loop().time()

        try:
            await rag_system.ingest_and_index(
                pdf_folder,
                f'benchmark_{config_name.lower()}.faiss',
                f'benchmark_{config_name.lower()}.pkl',
                limit=5,
                offset=0
            )

            end_time = asyncio.get_event_loop().time()
            total_time = end_time - start_time

            cache_stats = await rag_system.get_cache_stats()

            logger.info(f"{config_name} - Total time: {total_time:.2f}s")
            logger.info(f"{config_name} - Cache stats: {cache_stats}")

        except Exception as e:
            logger.error(f"Error in {config_name} benchmark: {e}")

if __name__ == '__main__':
    # Check if we're in a notebook environment (Jupyter/Colab)
    try:
        # Try to get the current event loop
        loop = asyncio.get_running_loop()
        # If we get here, we're in a notebook with an existing event loop
        import nest_asyncio
        nest_asyncio.apply()  # Allow nested event loops

        # Run main example
        asyncio.run(main())

        # Uncomment to run other examples:
        # asyncio.run(example_different_methods())
        # asyncio.run(performance_benchmark())

    except RuntimeError:
        # No event loop running, we can use asyncio.run() normally
        asyncio.run(main())
    except ImportError:
        # nest_asyncio not available, use alternative approach
        try:
            # Get the current event loop
            loop = asyncio.get_event_loop()
            # Run the main function
            loop.run_until_complete(main())
        except RuntimeError:
            # Create a new event loop if none exists
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            loop.run_until_complete(main())
            loop.close()

INFO:__main__:Cache stats: {'cached_files': 0, 'total_size_mb': 0.0, 'cache_folder': 'txt_cache_2', 'methods': {}}
INFO:__main__:Testing with 1 documents starting from offset 0
INFO:__main__:Using parsing method: pypdf2
INFO:__main__:Updated cache stats: {'cached_files': 0, 'total_size_mb': 0.0, 'cache_folder': 'txt_cache_2', 'methods': {}}


In [9]:
os.environ["HF_TOKEN"] = "REDACTED-HF-TOKEN"

NameError: name 'os' is not defined

In [28]:
! pip install langchain faiss-cpu pdfplumber numpy chainlit aiofiles sentence-transformers torch

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 412.1 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 641.8 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 319.4 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 2.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.1/102.1 MB 2.8 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 9.8 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 6.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 9.2 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 13.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.6/33.6 MB 6.6 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Dependencies (install via pip)
# pip install langchain faiss-cpu pdfplumber numpy chainlit aiofiles sentence-transformers torch

import os
import glob
import faiss
import pickle
import asyncio
import aiofiles
import numpy as np
from langchain.text_splitter import RecursiveCharacterTextSplitter
from typing import List, Tuple, Optional, Dict, Any
from concurrent.futures import ThreadPoolExecutor
import logging
from enum import Enum
from dataclasses import dataclass
from sentence_transformers import SentenceTransformer
import torch
from google.generativeai.client import
# PDF processing libraries
import pdfplumber

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- Configuration ---
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Configuration for async processing
MAX_CONCURRENT_PDFS = 1
MAX_CONCURRENT_CHUNKS = 10
CHUNK_SIZE = 500
CHUNK_OVERLAP = 200
CACHE_FOLDER = "txt_cache"

class PDFParsingMethod(Enum):
    """Enumeration of available PDF parsing methods."""
    PDFPLUMBER = "pdfplumber"
    GEMINI="gemini"

@dataclass
class PDFParsingConfig:
    """Configuration for PDF parsing methods."""
    method: PDFParsingMethod = PDFParsingMethod.PDFPLUMBER
    extract_tables: bool = True
    preserve_formatting: bool = True
    min_text_length: int = 100

@dataclass
class EmbeddingConfig:
    """Configuration for embedding models."""
    model_name: str = EMBEDDING_MODEL
    device: str = "auto"  # auto, cpu, cuda
    batch_size: int = 32
    max_seq_length: int = 512
    normalize_embeddings: bool = True
    cache_folder: str = "embedding_cache"

class AsyncPDFRAG:
    def __init__(self, max_concurrent_pdfs: int = MAX_CONCURRENT_PDFS,
                 max_concurrent_chunks: int = MAX_CONCURRENT_CHUNKS,
                 cache_folder: str = CACHE_FOLDER,
                 parsing_config: PDFParsingConfig = None,
                 embedding_config: EmbeddingConfig = None):
        self.max_concurrent_pdfs = max_concurrent_pdfs
        self.max_concurrent_chunks = max_concurrent_chunks
        self.cache_folder = cache_folder
        self.parsing_config = parsing_config or PDFParsingConfig()
        self.embedding_config = embedding_config or EmbeddingConfig()

        self.executor = ThreadPoolExecutor(max_workers=max_concurrent_pdfs)

        # Initialize embedding model
        self._init_embedding_model()

        # Create cache folders
        os.makedirs(self.cache_folder, exist_ok=True)
        os.makedirs(self.embedding_config.cache_folder, exist_ok=True)

    def _init_embedding_model(self):
        """Initialize the sentence transformer model."""
        try:
            # Determine device
            if self.embedding_config.device == "auto":
                device = "cuda" if torch.cuda.is_available() else "cpu"
            else:
                device = self.embedding_config.device

            logger.info(f"Initializing embedding model: {self.embedding_config.model_name} on {device}")

            # Initialize model
            self.embedding_model = SentenceTransformer(
                self.embedding_config.model_name,
                device=device,
                cache_folder=self.embedding_config.cache_folder
            )

            # Set max sequence length if specified
            if self.embedding_config.max_seq_length:
                self.embedding_model.max_seq_length = self.embedding_config.max_seq_length

            logger.info(f"Embedding model initialized successfully on {device}")

        except Exception as e:
            logger.error(f"Error initializing embedding model: {e}")
            raise

    def get_cache_path(self, pdf_path: str) -> str:
        """Generate cache file path for a given PDF."""
        pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
        return os.path.join(self.cache_folder, f"{pdf_name}_pdfplumber.txt")

    async def is_pdf_cached(self, pdf_path: str) -> bool:
        """Check if PDF has already been parsed and cached."""
        cache_path = self.get_cache_path(pdf_path)
        if not os.path.exists(cache_path):
            return False

        try:
            pdf_mtime = os.path.getmtime(pdf_path)
            cache_mtime = os.path.getmtime(cache_path)
            return cache_mtime > pdf_mtime
        except OSError:
            return False

    async def load_from_cache(self, pdf_path: str) -> Optional[str]:
        """Load parsed text from cache if available."""
        cache_path = self.get_cache_path(pdf_path)
        try:
            async with aiofiles.open(cache_path, 'r', encoding='utf-8') as f:
                content = await f.read()
                logger.info(f"Loaded cached text for {os.path.basename(pdf_path)}")
                return content
        except Exception as e:
            logger.error(f"Error loading cache for {pdf_path}: {e}")
            return None

    async def save_to_cache(self, pdf_path: str, content: str):
        """Save parsed text to cache."""
        cache_path = self.get_cache_path(pdf_path)
        try:
            async with aiofiles.open(cache_path, 'w', encoding='utf-8') as f:
                await f.write(content)
                logger.info(f"Cached parsed text for {os.path.basename(pdf_path)}")
        except Exception as e:
            logger.error(f"Error saving cache for {pdf_path}: {e}")

def parse_with_gemini(self, pdf_path: str) -> str:
        """
        Parses the PDF document by treating it as a multimodal input (image/document)
        using the Gemini API, extracting the complete text equivalent and structure.
        """
        if not self.gemini_client:
            logger.error("Gemini client not initialized. Cannot parse with Gemini.")
            return ""

        # Use a system instruction to guide the model's extraction behavior
        system_instruction = (
            "You are an expert document parser. Your goal is to extract ALL text, "
            "including headers, footers, and content from the provided PDF file. "
            "You MUST preserve the document's original structure. "
            "Use clear Markdown formatting (e.g., #, ## for headings, **bold** text, "
            "and Markdown tables) to represent the document's layout accurately. "
            "Output ONLY the extracted text and structure."
        )

        model_name = "gemini-1.5-flash"
        parsed_text = ""
        uploaded_file = None

        try:
            logger.info(f"Uploading PDF {os.path.basename(pdf_path)} for Gemini parsing...")

            # 1. Upload the PDF file. This is crucial for multimodal input.
            uploaded_file = self.gemini_client.files.upload(file=pdf_path)
            logger.info(f"File uploaded successfully. URI: {uploaded_file.uri}")

            # 2. Call the API with the uploaded file
            response = self.gemini_client.models.generate_content(
                model=model_name,
                contents=[
                    system_instruction,
                    uploaded_file
                ]
            )

            parsed_text = response.text
            logger.info(f"Gemini parsing successful for {os.path.basename(pdf_path)}. Extracted {len(parsed_text)} characters.")

        except APIError as e:
            logger.error(f"Gemini API Error for {pdf_path}: {e}")
            return ""
        except Exception as e:
            logger.error(f"Error during Gemini parsing for {pdf_path}: {e}")
            return ""

        finally:
            # 3. Clean up: Delete the file from the API service
            if uploaded_file:
                try:
                    self.gemini_client.files.delete(name=uploaded_file.name)
                    logger.info(f"Deleted uploaded file: {uploaded_file.name}")
                except Exception as e:
                    logger.warning(f"Could not delete uploaded file {uploaded_file.name}: {e}")

        return parsed_text
    def parse_pdf_with_pdfplumber(self, pdf_path: str) -> str:
        """Parse PDF using pdfplumber library."""
        try:
            text_content = []

            with pdfplumber.open(pdf_path) as pdf:
                # Extract metadata
                metadata = pdf.metadata or {}
                if metadata:
                    text_content.append("=== DOCUMENT METADATA ===")
                    for key, value in metadata.items():
                        if value:
                            text_content.append(f"{key}: {value}")
                    text_content.append("")

                # Extract text from each page
                for page_num, page in enumerate(pdf.pages, 1):
                    page_text = page.extract_text()
                    if page_text and page_text.strip():
                        text_content.append(f"=== PAGE {page_num} ===")
                        text_content.append(page_text.strip())
                        text_content.append("")

                    # Extract tables if configured
                    if self.parsing_config.extract_tables:
                        tables = page.extract_tables()
                        for table_num, table in enumerate(tables, 1):
                            text_content.append(f"--- Table {table_num} on Page {page_num} ---")
                            for row in table:
                                if row:
                                    text_content.append(" | ".join(str(cell) if cell else "" for cell in row))
                            text_content.append("")

            return "\n".join(text_content)

        except Exception as e:
            logger.error(f"Error parsing PDF with pdfplumber {pdf_path}: {e}")
            return ""

    def evaluate_parsing_quality(self, text: str, pdf_path: str) -> Dict[str, Any]:
        """Evaluate the quality of parsed text."""
        if not text:
            return {"score": 0, "reason": "No text extracted"}

        text_length = len(text.strip())
        word_count = len(text.split())
        line_count = len(text.split('\n'))

        has_structure = any(marker in text for marker in ['===', '---', '#', '##'])
        has_metadata = 'METADATA' in text or 'metadata' in text.lower()

        score = 0
        reasons = []

        if text_length >= self.parsing_config.min_text_length:
            score += 40
        else:
            reasons.append(f"Text too short ({text_length} chars)")

        if word_count > 50:
            score += 20

        if has_structure:
            score += 20

        if has_metadata:
            score += 10

        if line_count > 10:
            score += 10

        return {
            "score": score,
            "text_length": text_length,
            "word_count": word_count,
            "line_count": line_count,
            "has_structure": has_structure,
            "has_metadata": has_metadata,
            "reasons": reasons
        }

    async def parse_pdf(self, pdf_path: str) -> str:
        """Parse PDF using pdfplumber with caching."""
        # Check cache first
        if await self.is_pdf_cached(pdf_path):
            cached_content = await self.load_from_cache(pdf_path)
            if cached_content:
                return cached_content

        logger.info(f"Parsing {os.path.basename(pdf_path)} with pdfplumber")

        # Parse with pdfplumber
        loop = asyncio.get_event_loop()
        parsed_text = await loop.run_in_executor(self.executor, self.parse_pdf_with_pdfplumber, pdf_path)

        # Cache the result if parsing was successful
        if parsed_text:
            await self.save_to_cache(pdf_path, parsed_text)

        return parsed_text

    def chunk_text(self, full_text: str) -> List[str]:
        """Split text into semantically coherent chunks."""
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", " ", ""],
            length_function=len
        )
        return splitter.split_text(full_text)

    async def embed_chunks_batch(self, chunks: List[str]) -> np.ndarray:
        """Embed chunks using all-mpnet-base-v2 sentence transformer model."""
        if not chunks:
            return np.array([])

        def _embed_batch(chunk_batch):
            try:
                # Encode chunks using sentence transformer
                embeddings = self.embedding_model.encode(
                    chunk_batch,
                    batch_size=self.embedding_config.batch_size,
                    show_progress_bar=False,
                    convert_to_numpy=True,
                    normalize_embeddings=self.embedding_config.normalize_embeddings
                )
                return embeddings
            except Exception as e:
                logger.error(f"Error embedding batch: {e}")
                return np.array([])

        # Process in batches to manage memory
        batch_size = self.embedding_config.batch_size
        batches = [chunks[i:i + batch_size] for i in range(0, len(chunks), batch_size)]

        logger.info(f"Embedding {len(chunks)} chunks in {len(batches)} batches using {self.embedding_config.model_name}")

        # Run batches concurrently
        loop = asyncio.get_event_loop()
        tasks = [loop.run_in_executor(self.executor, _embed_batch, batch) for batch in batches]

        embeddings_batches = await asyncio.gather(*tasks)

        # Concatenate all embeddings
        all_embeddings = []
        for batch_embeddings in embeddings_batches:
            if len(batch_embeddings) > 0:
                all_embeddings.append(batch_embeddings)

        if all_embeddings:
            return np.vstack(all_embeddings).astype("float32")
        else:
            return np.array([])

    async def build_faiss_index(self, chunks: List[str]) -> Tuple[faiss.Index, List[str]]:
        """Build FAISS index asynchronously."""
        if not chunks:
            raise ValueError("No chunks provided for indexing")

        embeddings_np = await self.embed_chunks_batch(chunks)

        if len(embeddings_np) == 0:
            raise ValueError("Failed to generate embeddings")

        # Build the FAISS index
        dim = embeddings_np.shape[1]
        index = faiss.IndexFlatL2(dim)
        index.add(embeddings_np)

        logger.info(f"FAISS index built successfully with {len(chunks)} chunks, embedding dim: {dim}")
        return index, chunks.copy()

    async def save_index(self, index: faiss.Index, metadata: List[str],
                        index_path: str, meta_path: str):
        """Save FAISS index and metadata asynchronously."""
        def _save_index():
            faiss.write_index(index, index_path)

        def _save_metadata():
            with open(meta_path, "wb") as f:
                pickle.dump(metadata, f)

        loop = asyncio.get_event_loop()
        await asyncio.gather(
            loop.run_in_executor(self.executor, _save_index),
            loop.run_in_executor(self.executor, _save_metadata)
        )

        logger.info(f"Index saved to {index_path} and metadata to {meta_path}")

    async def load_index(self, index_path: str, meta_path: str) -> Tuple[faiss.Index, List[str]]:
        """Load FAISS index and metadata asynchronously."""
        def _load_index():
            return faiss.read_index(index_path)

        def _load_metadata():
            with open(meta_path, "rb") as f:
                return pickle.load(f)

        loop = asyncio.get_event_loop()
        index, metadata = await asyncio.gather(
            loop.run_in_executor(self.executor, _load_index),
            loop.run_in_executor(self.executor, _load_metadata)
        )

        logger.info("Index and metadata loaded successfully.")
        return index, metadata

    async def query_index(self, query: str, index: faiss.Index,
                         metadata: List[str], top_k: int = 10) -> List[str]:
        """Query the index asynchronously using all-mpnet-base-v2 embedding model."""
        def _embed_query():
            return self.embedding_model.encode(
                [query],
                convert_to_numpy=True,
                normalize_embeddings=self.embedding_config.normalize_embeddings
            )[0]

        def _search_index(q_emb):
            distances, indices = index.search(np.array([q_emb], dtype="float32"), top_k)
            return [metadata[i] for i in indices[0]]

        loop = asyncio.get_event_loop()
        q_emb = await loop.run_in_executor(self.executor, _embed_query)
        results = await loop.run_in_executor(self.executor, _search_index, q_emb)

        return results

    def select_pdf_files(self, pdf_files: List[str], limit: Optional[int] = None,
                        offset: int = 0) -> List[str]:
        """Select a subset of PDF files based on offset and limit."""
        total_files = len(pdf_files)

        if offset >= total_files:
            logger.warning(f"Offset {offset} is >= total files {total_files}. No files selected.")
            return []

        sorted_files = sorted(pdf_files)
        selected_files = sorted_files[offset:]

        if limit is not None:
            selected_files = selected_files[:limit]

        logger.info(f"Selected {len(selected_files)} files (offset: {offset}, limit: {limit}) from {total_files} total files")

        for i, file in enumerate(selected_files):
            logger.info(f"  {i+1}. {os.path.basename(file)}")

        return selected_files

    async def process_pdf_batch(self, pdf_files: List[str]) -> List[str]:
        """Process multiple PDFs concurrently using pdfplumber."""
        semaphore = asyncio.Semaphore(self.max_concurrent_pdfs)

        async def _process_single_pdf(pdf_file):
            async with semaphore:
                structured_text = await self.parse_pdf(pdf_file)
                if structured_text:
                    return self.chunk_text(structured_text)
                return []

        tasks = [_process_single_pdf(pdf_file) for pdf_file in pdf_files]
        results = await asyncio.gather(*tasks, return_exceptions=True)

        all_chunks = []
        processed_count = 0

        for i, result in enumerate(results):
            if isinstance(result, Exception):
                logger.error(f"Error processing PDF {pdf_files[i]}: {result}")
            elif result:
                all_chunks.extend(result)
                processed_count += 1

        logger.info(f"Successfully processed {processed_count} PDFs")
        return all_chunks

    async def ingest_and_index(self, pdf_folder: str, index_path: str, meta_path: str,
                              limit: Optional[int] = None, offset: int = 0):
        """Complete ingestion pipeline using pdfplumber and all-mpnet-base-v2."""
        all_pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))
        if not all_pdf_files:
            logger.warning(f"No PDF files found in '{pdf_folder}'.")
            return

        pdf_files = self.select_pdf_files(all_pdf_files, limit=limit, offset=offset)

        if not pdf_files:
            logger.warning("No PDF files selected for processing.")
            return

        logger.info(f"Processing {len(pdf_files)} PDF files with pdfplumber")
        logger.info(f"Using embedding model: {self.embedding_config.model_name}")

        all_chunks = await self.process_pdf_batch(pdf_files)

        if all_chunks:
            logger.info(f"Generated {len(all_chunks)} chunks from {len(pdf_files)} PDFs")
            index, metadata = await self.build_faiss_index(all_chunks)
            await self.save_index(index, metadata, index_path, meta_path)
        else:
            logger.warning("No text was extracted from the PDFs. Index not built.")

    async def clear_cache(self):
        """Clear all cached text files."""
        try:
            import shutil
            if os.path.exists(self.cache_folder):
                shutil.rmtree(self.cache_folder)
                os.makedirs(self.cache_folder, exist_ok=True)
                logger.info("Cache cleared successfully")
        except Exception as e:
            logger.error(f"Error clearing cache: {e}")

    async def get_cache_stats(self) -> dict:
        """Get statistics about cached files."""
        try:
            cache_files = glob.glob(os.path.join(self.cache_folder, "*.txt"))
            total_size = sum(os.path.getsize(f) for f in cache_files)

            return {
                "cached_files": len(cache_files),
                "total_size_mb": total_size / (1024 * 1024),
                "cache_folder": self.cache_folder,
                "embedding_model": self.embedding_config.model_name,
                "embedding_device": str(self.embedding_model.device),
                "parsing_method": "pdfplumber"
            }
        except Exception as e:
            logger.error(f"Error getting cache stats: {e}")
            return {"error": str(e)}

    def __del__(self):
        """Cleanup executor on destruction."""
        if hasattr(self, 'executor'):
            self.executor.shutdown(wait=True)


# --- Example Usage ---
async def main():
    # Configuration
    PDF_FOLDER_PATH = 'pdfs_20'
    INDEX_FILE_PATH = 'index.faiss'
    METADATA_FILE_PATH = 'meta.pkl'
    CACHE_FOLDER = 'txt_cache'

    # Test configuration
    TEST_LIMIT = 5
    TEST_OFFSET = 0

    # Configure parsing and embedding
    parsing_config = PDFParsingConfig(
        method=PDFParsingMethod.PDFPLUMBER,
        extract_tables=True,
        preserve_formatting=True,
        min_text_length=100
    )

    embedding_config = EmbeddingConfig(
        model_name=EMBEDDING_MODEL,
        device="auto",
        batch_size=32,
        normalize_embeddings=True
    )

    # Create the PDF folder if it doesn't exist
    if not os.path.exists(PDF_FOLDER_PATH):
        os.makedirs(PDF_FOLDER_PATH)
        logger.info(f"Created folder '{PDF_FOLDER_PATH}'. Please add your PDF files there.")
        return

    # Initialize the RAG system
    rag_system = AsyncPDFRAG(
        cache_folder=CACHE_FOLDER,
        parsing_config=parsing_config,
        embedding_config=embedding_config
    )

    try:
        # Show cache statistics
        cache_stats = await rag_system.get_cache_stats()
        logger.info(f"Cache stats: {cache_stats}")

        # Run the ingestion and indexing pipeline
        logger.info(f"Testing with {TEST_LIMIT} documents starting from offset {TEST_OFFSET}")
        logger.info(f"Using pdfplumber for PDF parsing")
        logger.info(f"Using {EMBEDDING_MODEL} for embeddings")

        await rag_system.ingest_and_index(
            PDF_FOLDER_PATH,
            INDEX_FILE_PATH,
            METADATA_FILE_PATH,
            limit=TEST_LIMIT,
            offset=TEST_OFFSET
        )

        # Show updated cache statistics
        cache_stats = await rag_system.get_cache_stats()
        logger.info(f"Updated cache stats: {cache_stats}")

        # Example query after indexing
        if os.path.exists(INDEX_FILE_PATH):
            index, metadata = await rag_system.load_index(INDEX_FILE_PATH, METADATA_FILE_PATH)

            # Test query
            test_query = "What is the main topic of the documents? Generate a summary for the document content."
            results = await rag_system.query_index(test_query, index, metadata)

            logger.info(f"Query: {test_query}")
            logger.info(f"Results: {len(results)} chunks retrieved")

            # Show sample results
            for i, result in enumerate(results[:2]):
                logger.info(f"Result {i+1}: {result[:200]}...")

    except Exception as e:
        logger.error(f"Error in main pipeline: {e}")

# --- Performance Testing ---
async def performance_test():
    """Test performance with different batch sizes and configurations."""

    configs = [
        {"batch_size": 16, "max_concurrent": 2},
        {"batch_size": 32, "max_concurrent": 3},
        {"batch_size": 64, "max_concurrent": 4}
    ]

    for config in configs:
        logger.info(f"\n=== Testing config: {config} ===")

        embedding_config = EmbeddingConfig(
            model_name=EMBEDDING_MODEL,
            batch_size=config["batch_size"]
        )

        rag_system = AsyncPDFRAG(
            max_concurrent_pdfs=config["max_concurrent"],
            cache_folder=f'test_cache_{config["batch_size"]}',
            embedding_config=embedding_config
        )

        start_time = asyncio.get_event_loop().time()

        try:
            await rag_system.ingest_and_index(
                '/content/pdfs_20',
                f'test_index_{config["batch_size"]}.faiss',
                f'test_meta_{config["batch_size"]}.pkl',
                limit=2,
                offset=0
            )

            end_time = asyncio.get_event_loop().time()
            total_time = end_time - start_time

            logger.info(f"Config {config} - Total time: {total_time:.2f}s")

        except Exception as e:
            logger.error(f"Error in performance test: {e}")

if __name__ == '__main__':
    # Check if we're in a notebook environment (Jupyter/Colab)
    try:
        # Try to get the current event loop
        loop = asyncio.get_running_loop()
        # If we get here, we're in a notebook with an existing event loop
        import nest_asyncio
        nest_asyncio.apply()  # Allow nested event loops

        # Run main example
        asyncio.run(main())

        # Uncomment to run performance test:
        # asyncio.run(performance_test())

    except RuntimeError:
        # No event loop running, we can use asyncio.run() normally
        asyncio.run(main())
    except ImportError:
        # nest_asyncio not available, use alternative approach
        try:
            # Get the current event loop
            loop = asyncio.get_event_loop()
            # Run the main function
            loop.run_until_complete(main())
        except RuntimeError:
            # Create a new event loop if none exists
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            loop.run_until_complete(main())
            loop.close()

INFO:__main__:Initializing embedding model: sentence-transformers/all-MiniLM-L6-v2 on cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:__main__:Embedding model initialized successfully on cpu
INFO:__main__:Cache stats: {'cached_files': 5, 'total_size_mb': 4.754748344421387, 'cache_folder': 'txt_cache', 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'embedding_device': 'cpu', 'parsing_method': 'pdfplumber'}
INFO:__main__:Testing with 5 documents starting from offset 0
INFO:__main__:Using pdfplumber for PDF parsing
INFO:__main__:Using sentence-transformers/all-MiniLM-L6-v2 for embeddings
INFO:__main__:Selected 5 files (offset: 0, limit: 5) from 273 total files
INFO:__main__:  1. 01_gcf-b42-02-add17-funding-proposal-package-fp275.pdf
INFO:__main__:  2. 02_gcf-b42-02-add16-funding-proposal-package-fp274.pdf
INFO:__main__:  3. 03_gcf-b42-02-add15-funding-proposal-package-fp273.pdf
INFO:__main__:

In [11]:
import base64
import google.generativeai as genai

# 1. Configure Gemini with your API key
genai.configure(api_key=GEMINI_API_KEY)

# 2. Encode PDF as image-like bytes (each page rendered as image)
# If you want to treat the whole PDF like an image blob, you can read raw bytes
with open("pdfs_20_2/99_gcf-b30-02-add08.pdf", "rb") as f:
    pdf_bytes = f.read()

# Gemini expects inline data for image-like inputs
pdf_data = {
    "mime_type": "application/pdf",  # still supported, but we can treat as "image/*"
    "data": base64.b64encode(pdf_bytes).decode("utf-8")
}

# 3. Initialize a multimodal model
model = genai.GenerativeModel("gemini-1.5-pro")  # or gemini-1.5-pro for better quality

# 4. Ask Gemini to extract text
response = model.generate_content(
    [
        "Extract all the text content from this PDF document and return it as plain TXT format.",
        pdf_data
    ]
)

# 5. Get result
print(response.text)

# Optionally, write to txt file
with open("output.txt", "w", encoding="utf-8") as out:
    out.write(response.text)


E0000 00:00:1758900112.682506    3355 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0
Please retry in 50.108132852s. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-1.5-pro"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
}
violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerM

In [12]:
!pip install openai


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
OPEN_AI_API_KEY="REDACTED-OPENAI-KEY"

In [ ]:
import os
import re
import math
from openai import OpenAI

# ---------- Config ----------
MODEL = "gpt-4o"          # or "gpt-4o-mini" (cheaper), "gpt-4.1" (more reasoning)
PDF_PATH = "pdfs_20_2/99_gcf-b30-02-add08.pdf"
OUT_MD = "document.md"
CHUNK_DIR = "chunks"       # folder to save chunked markdown
TARGET_CHARS = 3500        # ~2–3k tokens per chunk once embedded (tune as needed)

# ---------- Client ----------
client = OpenAI(api_key=OPEN_AI_API_KEY)

# ---------- 1) Upload PDF ----------
uploaded = client.files.create(
    file=open(PDF_PATH, "rb"),
    purpose="assistants"  # valid for models to read
)

# ---------- 2) Ask for Markdown extraction ----------
# We’ll use Chat Completions with a file attachment. Content types allowed here include "text" and "file".
# Prompt is crafted to yield RAG-friendly, clean Markdown.
system_msg = {
    "role": "system",
    "content": (
        "You are a meticulous Markdown converter. "
        "Extract ALL usable text from the attached PDF and produce clean, GitHub-flavored Markdown. "
        "Rules:\n"
        "1) Preserve the document structure with proper #/##/### headings.\n"
        "2) Convert tables into Markdown tables.\n"
        "3) Preserve lists, footnotes, captions.\n"
        "4) Keep URLs as explicit markdown links when present.\n"
        "5) Insert a page marker as: `\n\n---\n*Page {n}*\n---\n` between pages when you detect page boundaries.\n"
        "6) Do NOT summarize—output all text verbatim where possible.\n"
        "7) Avoid OCR noise; remove repeated headers/footers; keep only meaningful content.\n"
        "8) Output ONLY Markdown. No extra commentary."
    )
}

user_msg = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Convert this PDF to Markdown for RAG ingestion."},
        {"type": "file", "file_id": uploaded.id}
    ]
}

resp = client.chat.completions.create(
    model=MODEL,
    messages=[system_msg, user_msg],
    # If your PDFs are long, you may want to raise this:
    max_completion_tokens=4096,  # tune based on size/cost limits
    temperature=0.0
)

# ---------- 3) Save single Markdown ----------
content = resp.choices[0].message.content
with open(OUT_MD, "w", encoding="utf-8") as f:
    f.write(content or "")

print(f"Saved Markdown to {OUT_MD}")

# ---------- 4) (Optional) Chunk the Markdown for RAG ----------
# Simple chunker: split on headings first to keep semantic boundaries,
# then pack into ~TARGET_CHARS chunks.

os.makedirs(CHUNK_DIR, exist_ok=True)

# Split on H1/H2/H3 while preserving the headings in the output
parts = re.split(r"(^#{1,3}\s.+?$)", content or "", flags=re.MULTILINE)
# parts looks like ["pre", "## Heading", "section text", "## Next", "text", ...]

# Re-join into section blocks: [("## Heading", "text..."), ...]
sections = []
i = 0
while i < len(parts):
    if re.match(r"^#{1,3}\s", parts[i] or "", flags=re.MULTILINE):
        heading = parts[i]
        body = parts[i+1] if i+1 < len(parts) else ""
        sections.append(heading + "\n" + body)
        i += 2
    else:
        # leading preface before first heading
        if parts[i].strip():
            sections.append(parts[i])
        i += 1

# Pack sections into size-limited chunks
chunks = []
buf = []
buf_len = 0
for sec in sections:
    s = sec.strip() + "\n\n"
    if buf_len + len(s) > TARGET_CHARS and buf:
        chunks.append("".join(buf))
        buf, buf_len = [], 0
    buf.append(s)
    buf_len += len(s)

if buf:
    chunks.append("".join(buf))

# Write chunk files
for idx, ch in enumerate(chunks, start=1):
    path = os.path.join(CHUNK_DIR, f"chunk_{idx:03d}.md")
    with open(path, "w", encoding="utf-8") as f:
        f.write(ch)
print(f"Wrote {len(chunks)} chunks to ./{CHUNK_DIR}")



BadRequestError: Error code: 400 - {'error': {'message': "Missing required parameter: 'messages[1].content[1].file'.", 'type': 'invalid_request_error', 'param': 'messages[1].content[1].file', 'code': 'missing_required_parameter'}}

In [20]:
import os
from openai import OpenAI

MODEL = "gpt-4o"
PDF_PATH = "pdfs_20_2/99_gcf-b30-02-add08.pdf"
OUT_MD = "document.md"

client = OpenAI(api_key=OPEN_AI_API_KEY)

# 1) Upload
uploaded = client.files.create(
    file=open(PDF_PATH, "rb"),
    purpose="assistants",
)

# 2) Create a single Responses call with an input file item
resp = client.responses.create(
    model=MODEL,
    input=[
        {
            "role": "system",
            "content": "You convert PDFs into clean, GitHub-flavored Markdown for RAG. Preserve headings, lists, tables, links. No summaries.",
        },
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "Convert this PDF to Markdown."},
                {"type": "input_file", "file_id": uploaded.id},  # <-- Responses API shape
            ],
        },
    ],
    temperature=0,
    # response_format={"type": "text"}  # default text output
)

# 3) Extract plain text from Responses output (Items)
def responses_output_text(r):
    # Prefer official helper if present; otherwise flatten text items
    if hasattr(r, "output_text") and r.output_text:
        return r.output_text
    chunks = []
    for out in getattr(r, "output", []) or []:
        for c in getattr(out, "content", []) or []:
            if getattr(c, "type", None) in ("output_text", "text"):
                chunks.append(getattr(c, "text", "") or "")
    return "".join(chunks)

md_text = responses_output_text(resp)

with open(OUT_MD, "w", encoding="utf-8") as f:
    f.write(md_text or "")

print(f"✅ Saved Markdown to {OUT_MD}")


BadRequestError: Error code: 400 - {'error': {'message': 'Your input exceeds the context window of this model. Please adjust your input and try again.', 'type': 'invalid_request_error', 'param': 'input', 'code': 'context_length_exceeded'}}

In [21]:
!pip install pypdf

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 585.0 kB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [4]:
#!/usr/bin/env python3
import os
import tempfile
from typing import List
from openai import OpenAI

try:
    from pypdf import PdfReader, PdfWriter  # pip install pypdf
except ImportError:
    raise SystemExit("Please: pip install pypdf")

# ------------- CONFIG -------------
MODEL = "gpt-4o"            # or "gpt-4o-mini" (cheaper); "gpt-4.1" (more reasoning)
PDF_PATH = "pdfs_20_2/01_gcf-b42-02-add17-funding-proposal-package-fp275.pdf"
OUT_MD = "01_gcf-b42-02-add17-funding-proposal-package-fp275.pdf"
PAGES_PER_CHUNK = 8         # lower if you still hit context limits
TEMPERATURE = 0.0
# ----------------------------------

api_key = OPEN_AI_API_KEY
if not api_key:
    raise SystemExit("Set OPENAI_API_KEY in your environment.")
client = OpenAI(api_key=api_key)

def split_pdf_to_chunks(pdf_path: str, pages_per_chunk: int) -> List[str]:
    """Return a list of temp PDF file paths, each containing <= pages_per_chunk."""
    reader = PdfReader(pdf_path)
    n_pages = len(reader.pages)
    temp_files = []

    for start in range(0, n_pages, pages_per_chunk):
        end = min(start + pages_per_chunk, n_pages)
        writer = PdfWriter()
        for i in range(start, end):
            writer.add_page(reader.pages[i])
        fd, tmp_pdf = tempfile.mkstemp(suffix=f".{start+1}-{end}.pdf")
        os.close(fd)
        with open(tmp_pdf, "wb") as f:
            writer.write(f)
        temp_files.append(tmp_pdf)

    return temp_files

def extract_output_text(resp) -> str:
    """Flatten Responses API output to a single text string."""
    if hasattr(resp, "output_text") and resp.output_text:
        return resp.output_text
    chunks = []
    for out in getattr(resp, "output", []) or []:
        for c in getattr(out, "content", []) or []:
            if getattr(c, "type", None) in ("output_text", "text"):
                chunks.append(getattr(c, "text", "") or "")
    return "".join(chunks)

def convert_pdf_chunk_to_markdown(chunk_pdf_path: str, page_range_label: str) -> str:
    # 1) Upload this chunk
    uploaded = client.files.create(
        file=open(chunk_pdf_path, "rb"),
        purpose="assistants",
    )

    # 2) Ask model to convert to Markdown
    resp = client.responses.create(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": (
                    "You convert PDFs into clean, GitHub-flavored Markdown for RAG. "
                    "Rules:\n"
                    "1) Preserve structure with proper #/##/### headings.\n"
                    "2) Convert tables to Markdown tables.\n"
                    "3) Keep lists and links.\n"
                    "4) Remove repeated headers/footers and OCR noise.\n"
                    "5) No summaries—output the full text you can read.\n"
                    "6) Output ONLY Markdown (no extra commentary)."
                ),
            },
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": f"Convert this PDF chunk ({page_range_label}) to Markdown."},
                    {"type": "input_file", "file_id": uploaded.id},
                ],
            },
        ],
        temperature=TEMPERATURE,
    )

    md = extract_output_text(resp)
    # Add a clear marker for the chunk boundaries
    return f"\n\n---\n*Chunk {page_range_label}*\n---\n\n{md.strip()}\n"

def main():
    # Split the big PDF into smaller PDFs
    chunk_paths = split_pdf_to_chunks(PDF_PATH, PAGES_PER_CHUNK)

    all_md_parts = []
    for chunk_path in chunk_paths:
        # derive label like "pages 1-8" from temp file name suffix
        label = os.path.basename(chunk_path).split(".")[-2]  # e.g., "1-8"
        page_range_label = f"pages {label.replace('-', '–')}"
        try:
            part_md = convert_pdf_chunk_to_markdown(chunk_path, page_range_label)
            all_md_parts.append(part_md)
            print (f"✅ Processed {page_range_label}")
            print(f"---\n{part_md[:500]}...\n---\n")  # print first 500 chars
        except Exception as e:
            print(f"⚠️ Failed on {page_range_label}: {e}")
        finally:
            # cleanup temp file
            try:
                os.remove(chunk_path)
            except OSError:
                pass

    full_md = "".join(all_md_parts).strip() + "\n"
    with open(OUT_MD, "w", encoding="utf-8") as f:
        f.write(full_md)

    print(f"✅ Wrote Markdown to {OUT_MD} ({len(all_md_parts)} chunks).")
    if not all_md_parts:
        print("No content extracted. Try lowering PAGES_PER_CHUNK.")

if __name__ == "__main__":
    main()


✅ Processed pages 1–8
---


---
*Chunk pages 1–8*
---

```markdown
# Consideration of funding proposals – Addendum XVII

## Funding proposal package for FP275

### Summary

This addendum contains the following six parts:
a) A funding proposal summary titled “Scaling up the Deployment of Integrated Utility Services (IUS) to Support Energy Sector Transformation in the Caribbean (Phase 1) Programme” by Caribbean Development Bank;
b) No-objection letter(s) issued by the national designated authority(ies) or focal point(s);
c...
---

✅ Processed pages 9–16
---


---
*Chunk pages 9–16*
---

```markdown
# GREEN CLIMATE FUND FUNDING PROPOSAL V.3.0

## Climate Context

### Climate Change Problem

#### Mitigation Needs

**B.1.1.** Barbados, Belize, and Jamaica heavily rely on imported fossil fuels for electricity, contributing significantly to their GHG emissions. The energy sectors in the BMCs that are participating in Phase I are characterized by a continued dependence on imported fossil fuels

In [ ]:
#!/usr/bin/env python3
import os
import tempfile
import asyncio
from pathlib import Path
from typing import List, Tuple, Dict
from dataclasses import dataclass
import aiohttp
import json

try:
    from pypdf import PdfReader, PdfWriter  # pip install pypdf
except ImportError:
    raise SystemExit("Please: pip install pypdf")

try:
    import aiofiles  # pip install aiofiles
except ImportError:
    raise SystemExit("Please: pip install aiofiles")

# ------------- CONFIG -------------
MODEL = "gpt-4o"            # or "gpt-4o-mini" (cheaper); "gpt-4.1" (more reasoning)
FOLDER_PATH = "pdfs_20_2"   # Folder containing PDFs
OUTPUT_DIR = "output_md"    # Output directory for markdown files
PAGES_PER_CHUNK = 8         # lower if you still hit context limits
TEMPERATURE = 0.0
MAX_CONCURRENT = 10         # Maximum concurrent coroutines (can be higher than threads)
CHUNK_SIZE = 50             # Process chunks in batches to avoid overwhelming the API
# ----------------------------------

@dataclass
class ChunkInfo:
    pdf_name: str
    chunk_index: int
    chunk_path: str
    page_range_label: str

class AsyncPDFProcessor:
    def __init__(self, api_key: str):
        if not api_key:
            raise SystemExit("Set OPENAI_API_KEY in your environment.")
        self.api_key = api_key
        self.base_url = "https://api.openai.com/v1"

    def get_pdf_files(self, folder_path: str) -> List[str]:
        """Get all PDF files from the specified folder."""
        folder = Path(folder_path)
        if not folder.exists():
            raise SystemExit(f"Folder {folder_path} does not exist.")

        pdf_files = list(folder.glob("*.pdf"))
        if not pdf_files:
            raise SystemExit(f"No PDF files found in {folder_path}")

        # Sort to ensure consistent processing order
        return sorted([str(pdf) for pdf in pdf_files])

    def split_pdf_to_chunks(self, pdf_path: str, pages_per_chunk: int) -> List[ChunkInfo]:
        """Return a list of ChunkInfo objects for each chunk."""
        reader = PdfReader(pdf_path)
        n_pages = len(reader.pages)
        chunks = []
        pdf_name = Path(pdf_path).stem

        for chunk_idx, start in enumerate(range(0, n_pages, pages_per_chunk)):
            end = min(start + pages_per_chunk, n_pages)
            writer = PdfWriter()
            for i in range(start, end):
                writer.add_page(reader.pages[i])

            fd, tmp_pdf = tempfile.mkstemp(
                suffix=f".{pdf_name}.chunk{chunk_idx:03d}.{start+1}-{end}.pdf"
            )
            os.close(fd)

            with open(tmp_pdf, "wb") as f:
                writer.write(f)

            page_range_label = f"pages {start+1}–{end}"
            chunks.append(ChunkInfo(
                pdf_name=pdf_name,
                chunk_index=chunk_idx,
                chunk_path=tmp_pdf,
                page_range_label=page_range_label
            ))

        return chunks

    async def upload_file_async(self, session: aiohttp.ClientSession, file_path: str) -> str:
        """Upload file to OpenAI and return file ID."""
        async with aiofiles.open(file_path, 'rb') as f:
            file_content = await f.read()

        data = aiohttp.FormData()
        data.add_field('file', file_content, filename=os.path.basename(file_path))
        data.add_field('purpose', 'assistants')

        headers = {"Authorization": f"Bearer {self.api_key}"}

        async with session.post(f"{self.base_url}/files", data=data, headers=headers) as response:
            if response.status != 200:
                error_text = await response.text()
                raise Exception(f"File upload failed: {response.status} - {error_text}")

            result = await response.json()
            return result['id']

    async def convert_chunk_async(self, session: aiohttp.ClientSession, file_id: str, chunk_info: ChunkInfo) -> str:
        """Convert PDF chunk to markdown using OpenAI API."""
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": MODEL,
            "input": [
                {
                    "role": "system",
                    "content": (
                        "You convert PDFs into clean, GitHub-flavored Markdown for RAG. "
                        "Rules:\n"
                        "1) Preserve structure with proper #/##/### headings.\n"
                        "2) Convert tables to Markdown tables.\n"
                        "3) Keep lists and links.\n"
                        "4) Remove repeated headers/footers and OCR noise.\n"
                        "5) No summaries—output the full text you can read.\n"
                        "6) Output ONLY Markdown (no extra commentary)."
                    ),
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "input_text", "text": f"Convert this PDF chunk ({chunk_info.page_range_label}) to Markdown."},
                        {"type": "input_file", "file_id": file_id},
                    ],
                },
            ],
            "temperature": TEMPERATURE,
        }

        async with session.post(f"{self.base_url}/responses", json=payload, headers=headers) as response:
            if response.status != 200:
                error_text = await response.text()
                raise Exception(f"API request failed: {response.status} - {error_text}")

            result = await response.json()
            return self.extract_output_text_from_response(result)

    def extract_output_text_from_response(self, response_data: dict) -> str:
        """Extract text from API response."""
        # Handle different response formats
        if 'output_text' in response_data:
            return response_data['output_text']

        chunks = []
        output = response_data.get('output', [])
        for out in output:
            content = out.get('content', [])
            for c in content:
                if c.get('type') in ('output_text', 'text'):
                    chunks.append(c.get('text', ''))

        return ''.join(chunks)

    async def process_single_chunk(self, session: aiohttp.ClientSession, chunk_info: ChunkInfo, semaphore: asyncio.Semaphore) -> Tuple[ChunkInfo, str]:
        """Process a single chunk with concurrency control."""
        async with semaphore:
            try:
                # Upload file
                file_id = await self.upload_file_async(session, chunk_info.chunk_path)

                # Convert to markdown
                md = await self.convert_chunk_async(session, file_id, chunk_info)

                # Format markdown with chunk boundaries
                formatted_md = f"\n\n---\n*Chunk {chunk_info.page_range_label}*\n---\n\n{md.strip()}\n"

                print(f"✅ Processed {chunk_info.pdf_name} - {chunk_info.page_range_label}")
                return chunk_info, formatted_md

            except Exception as e:
                print(f"⚠️ Failed on {chunk_info.pdf_name} - {chunk_info.page_range_label}: {e}")
                return chunk_info, f"\n\n---\n*Error processing {chunk_info.page_range_label}: {e}*\n---\n\n"
            finally:
                # Cleanup temp file
                try:
                    os.remove(chunk_info.chunk_path)
                except OSError:
                    pass

    async def process_chunks_batch(self, chunks: List[ChunkInfo]) -> Dict[str, List[Tuple[int, str]]]:
        """Process chunks in batches using coroutines."""
        results_by_pdf = {}
        semaphore = asyncio.Semaphore(MAX_CONCURRENT)

        # Configure session with connection limits
        connector = aiohttp.TCPConnector(
            limit=MAX_CONCURRENT * 2,  # Total connection pool size
            limit_per_host=MAX_CONCURRENT,  # Connections per host
            ttl_dns_cache=300,  # DNS cache TTL
            use_dns_cache=True,
        )

        timeout = aiohttp.ClientTimeout(total=300)  # 5 minute timeout

        async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
            # Process chunks in batches to avoid overwhelming the API
            for i in range(0, len(chunks), CHUNK_SIZE):
                batch = chunks[i:i + CHUNK_SIZE]
                print(f"🚀 Processing batch {i//CHUNK_SIZE + 1}/{(len(chunks)-1)//CHUNK_SIZE + 1} ({len(batch)} chunks)")

                # Create coroutines for this batch
                tasks = [
                    self.process_single_chunk(session, chunk, semaphore)
                    for chunk in batch
                ]

                # Wait for all tasks in this batch to complete
                batch_results = await asyncio.gather(*tasks, return_exceptions=True)

                # Organize results
                for result in batch_results:
                    if isinstance(result, Exception):
                        print(f"⚠️ Batch processing error: {result}")
                        continue

                    chunk_info, markdown_content = result
                    if chunk_info.pdf_name not in results_by_pdf:
                        results_by_pdf[chunk_info.pdf_name] = []

                    results_by_pdf[chunk_info.pdf_name].append(
                        (chunk_info.chunk_index, markdown_content)
                    )

                # Small delay between batches to be respectful to the API
                if i + CHUNK_SIZE < len(chunks):
                    await asyncio.sleep(1)

        # Sort results by chunk index to maintain proper order
        for pdf_name in results_by_pdf:
            results_by_pdf[pdf_name].sort(key=lambda x: x[0])

        return results_by_pdf

    async def save_markdown_files_async(self, results_by_pdf: Dict[str, List[Tuple[int, str]]], output_dir: str):
        """Save markdown files asynchronously."""
        os.makedirs(output_dir, exist_ok=True)

        async def save_single_file(pdf_name: str, chunks: List[Tuple[int, str]]):
            # Combine all chunks for this PDF in order
            full_md_parts = [content for _, content in chunks]
            full_md = "".join(full_md_parts).strip() + "\n"

            output_path = os.path.join(output_dir, f"{pdf_name}.md")
            async with aiofiles.open(output_path, "w", encoding="utf-8") as f:
                await f.write(full_md)

            print(f"✅ Wrote {output_path} ({len(chunks)} chunks)")

        # Save all files concurrently
        tasks = [
            save_single_file(pdf_name, chunks)
            for pdf_name, chunks in results_by_pdf.items()
        ]
        await asyncio.gather(*tasks)

    async def process_folder_async(self, folder_path: str, output_dir: str):
        """Main async entry point to process all PDFs in a folder."""
        print(f"🔍 Scanning folder: {folder_path}")
        pdf_files = self.get_pdf_files(folder_path)
        print(f"📄 Found {len(pdf_files)} PDF files")

        # Split all PDFs into chunks (CPU-bound, keep synchronous)
        all_chunks = []
        for pdf_path in pdf_files:
            print(f"🔧 Splitting {Path(pdf_path).name} into chunks...")
            chunks = self.split_pdf_to_chunks(pdf_path, PAGES_PER_CHUNK)
            all_chunks.extend(chunks)

        print(f"📦 Total chunks to process: {len(all_chunks)}")
        print(f"🚀 Processing with {MAX_CONCURRENT} concurrent coroutines...")

        # Process all chunks asynchronously
        results_by_pdf = await self.process_chunks_batch(all_chunks)

        # Save results asynchronously
        await self.save_markdown_files_async(results_by_pdf, output_dir)

        print(f"🎉 Completed processing {len(pdf_files)} PDFs!")
        print(f"📁 Output saved to: {output_dir}")

def main():
    api_key = OPEN_AI_API_KEY
    processor = AsyncPDFProcessor(api_key)

    # Run the async function
    asyncio.run(processor.process_folder_async(FOLDER_PATH, OUTPUT_DIR))

if __name__ == "__main__":
    main()

RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
#!/usr/bin/env python3
import os
import tempfile
import asyncio
from pathlib import Path
from typing import List, Tuple, Dict
from dataclasses import dataclass
import aiohttp
import json
from enum import Enum

class ModelType(Enum):
    GPT_4O = "gpt-4o"
    GPT_4O_MINI = "gpt-4o-mini"
    GPT_4_1 = "gpt-4.1"

class OutputFolderEnum(Enum):
    OUTPUT_MD_O4 = "output_md/o4"
    OUTPUT_MD_O4_MINI = "output_md/o4-mini"
    OUTPUT_MD_4_1 = "output_md/4-1"
try:
    from pypdf import PdfReader, PdfWriter  # pip install pypdf
except ImportError:
    raise SystemExit("Please: pip install pypdf")

try:
    import aiofiles  # pip install aiofiles
except ImportError:
    raise SystemExit("Please: pip install aiofiles")

# ------------- CONFIG -------------
MODEL = ModelType.GPT_4O_MINI #"gpt-4o"            # or "gpt-4o-mini" (cheaper); "gpt-4.1" (more reasoning)
FOLDER_PATH = "pdfs_20_2"   # Folder containing PDFs
OUTPUT_DIR = OutputFolderEnum.OUTPUT_MD_O4_MINI#"output_md/o4-mini"    # Output directory for markdown files
PAGES_PER_CHUNK = 8         # lower if you still hit context limits
TEMPERATURE = 0.0
MAX_CONCURRENT = 10         # Maximum concurrent coroutines (can be higher than threads)
CHUNK_SIZE = 50             # Process chunks in batches to avoid overwhelming the API
# ----------------------------------

@dataclass
class ChunkInfo:
    pdf_name: str
    chunk_index: int
    chunk_path: str
    page_range_label: str

class AsyncPDFProcessor:
    def __init__(self, api_key: str):
        if not api_key:
            raise SystemExit("Set OPENAI_API_KEY in your environment.")
        self.api_key = api_key
        self.base_url = "https://api.openai.com/v1"

    def get_pdf_files(self, folder_path: str,limit:int) -> List[str]:
        """Get all PDF files from the specified folder."""
        folder = Path(folder_path)
        if not folder.exists():
            raise SystemExit(f"Folder {folder_path} does not exist.")

        pdf_files = list(folder.glob("*.pdf"))
        if limit>0:
            pdf_files=pdf_files[:limit]

        if not pdf_files:
            raise SystemExit(f"No PDF files found in {folder_path}")

        # Sort to ensure consistent processing order
        return sorted([str(pdf) for pdf in pdf_files])

    def split_pdf_to_chunks(self, pdf_path: str, pages_per_chunk: int) -> List[ChunkInfo]:
        """Return a list of ChunkInfo objects for each chunk."""
        reader = PdfReader(pdf_path)
        n_pages = len(reader.pages)
        chunks = []
        pdf_name = Path(pdf_path).stem

        for chunk_idx, start in enumerate(range(0, n_pages, pages_per_chunk)):
            end = min(start + pages_per_chunk, n_pages)
            writer = PdfWriter()
            for i in range(start, end):
                writer.add_page(reader.pages[i])

            fd, tmp_pdf = tempfile.mkstemp(
                suffix=f".{pdf_name}.chunk{chunk_idx:03d}.{start+1}-{end}.pdf"
            )
            os.close(fd)

            with open(tmp_pdf, "wb") as f:
                writer.write(f)

            page_range_label = f"pages {start+1}–{end}"
            chunks.append(ChunkInfo(
                pdf_name=pdf_name,
                chunk_index=chunk_idx,
                chunk_path=tmp_pdf,
                page_range_label=page_range_label
            ))

        return chunks

    async def upload_file_async(self, session: aiohttp.ClientSession, file_path: str) -> str:
        """Upload file to OpenAI and return file ID."""
        async with aiofiles.open(file_path, 'rb') as f:
            file_content = await f.read()

        data = aiohttp.FormData()
        data.add_field('file', file_content, filename=os.path.basename(file_path))
        data.add_field('purpose', 'assistants')

        headers = {"Authorization": f"Bearer {self.api_key}"}

        async with session.post(f"{self.base_url}/files", data=data, headers=headers) as response:
            if response.status != 200:
                error_text = await response.text()
                raise Exception(f"File upload failed: {response.status} - {error_text}")

            result = await response.json()
            return result['id']

    async def convert_chunk_async(self, session: aiohttp.ClientSession, file_id: str, chunk_info: ChunkInfo) -> str:
        """Convert PDF chunk to markdown using OpenAI API."""
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": MODEL,
            "input": [
                {
                    "role": "system",
                    "content": (
                        "You convert PDFs into clean, GitHub-flavored Markdown for RAG. "
                        "Rules:\n"
                        "1) Preserve structure with proper #/##/### headings.\n"
                        "2) Convert tables to Markdown tables.\n"
                        "3) Keep lists and links.\n"
                        "4) Remove repeated headers/footers and OCR noise.\n"
                        "5) No summaries—output the full text you can read.\n"
                        "6) Output ONLY Markdown (no extra commentary)."
                    ),
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "input_text", "text": f"Convert this PDF chunk ({chunk_info.page_range_label}) to Markdown."},
                        {"type": "input_file", "file_id": file_id},
                    ],
                },
            ],
            "temperature": TEMPERATURE,
        }

        async with session.post(f"{self.base_url}/responses", json=payload, headers=headers) as response:
            if response.status != 200:
                error_text = await response.text()
                raise Exception(f"API request failed: {response.status} - {error_text}")

            result = await response.json()
            return self.extract_output_text_from_response(result)

    def extract_output_text_from_response(self, response_data: dict) -> str:
        """Extract text from API response."""
        # Handle different response formats
        if 'output_text' in response_data:
            return response_data['output_text']

        chunks = []
        output = response_data.get('output', [])
        for out in output:
            content = out.get('content', [])
            for c in content:
                if c.get('type') in ('output_text', 'text'):
                    chunks.append(c.get('text', ''))

        return ''.join(chunks)

    async def process_single_chunk(self, session: aiohttp.ClientSession, chunk_info: ChunkInfo, semaphore: asyncio.Semaphore) -> Tuple[ChunkInfo, str]:
        """Process a single chunk with concurrency control."""
        async with semaphore:
            try:
                # Upload file
                file_id = await self.upload_file_async(session, chunk_info.chunk_path)

                # Convert to markdown
                md = await self.convert_chunk_async(session, file_id, chunk_info)

                # Format markdown with chunk boundaries
                formatted_md = f"\n\n---\n*Chunk {chunk_info.page_range_label}*\n---\n\n{md.strip()}\n"

                print(f"✅ Processed {chunk_info.pdf_name} - {chunk_info.page_range_label}")
                return chunk_info, formatted_md

            except Exception as e:
                print(f"⚠️ Failed on {chunk_info.pdf_name} - {chunk_info.page_range_label}: {e}")
                return chunk_info, f"\n\n---\n*Error processing {chunk_info.page_range_label}: {e}*\n---\n\n"
            finally:
                # Cleanup temp file
                try:
                    os.remove(chunk_info.chunk_path)
                except OSError:
                    pass

    async def process_chunks_batch(self, chunks: List[ChunkInfo]) -> Dict[str, List[Tuple[int, str]]]:
        """Process chunks in batches using coroutines."""
        results_by_pdf = {}
        semaphore = asyncio.Semaphore(MAX_CONCURRENT)

        # Configure session with connection limits
        connector = aiohttp.TCPConnector(
            limit=MAX_CONCURRENT * 2,  # Total connection pool size
            limit_per_host=MAX_CONCURRENT,  # Connections per host
            ttl_dns_cache=300,  # DNS cache TTL
            use_dns_cache=True,
        )

        timeout = aiohttp.ClientTimeout(total=300)  # 5 minute timeout

        async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
            # Process chunks in batches to avoid overwhelming the API
            for i in range(0, len(chunks), CHUNK_SIZE):
                batch = chunks[i:i + CHUNK_SIZE]
                print(f"🚀 Processing batch {i//CHUNK_SIZE + 1}/{(len(chunks)-1)//CHUNK_SIZE + 1} ({len(batch)} chunks)")

                # Create coroutines for this batch
                tasks = [
                    self.process_single_chunk(session, chunk, semaphore)
                    for chunk in batch
                ]

                # Wait for all tasks in this batch to complete
                batch_results = await asyncio.gather(*tasks, return_exceptions=True)

                # Organize results
                for result in batch_results:
                    if isinstance(result, Exception):
                        print(f"⚠️ Batch processing error: {result}")
                        continue

                    chunk_info, markdown_content = result
                    if chunk_info.pdf_name not in results_by_pdf:
                        results_by_pdf[chunk_info.pdf_name] = []

                    results_by_pdf[chunk_info.pdf_name].append(
                        (chunk_info.chunk_index, markdown_content)
                    )

                # Small delay between batches to be respectful to the API
                if i + CHUNK_SIZE < len(chunks):
                    await asyncio.sleep(1)

        # Sort results by chunk index to maintain proper order
        for pdf_name in results_by_pdf:
            results_by_pdf[pdf_name].sort(key=lambda x: x[0])

        return results_by_pdf

    async def save_markdown_files_async(self, results_by_pdf: Dict[str, List[Tuple[int, str]]], output_dir: str):
        """Save markdown files asynchronously."""
        os.makedirs(output_dir, exist_ok=True)

        async def save_single_file(pdf_name: str, chunks: List[Tuple[int, str]]):
            # Combine all chunks for this PDF in order
            full_md_parts = [content for _, content in chunks]
            full_md = "".join(full_md_parts).strip() + "\n"

            output_path = os.path.join(output_dir, f"{pdf_name}.md")
            async with aiofiles.open(output_path, "w", encoding="utf-8") as f:
                await f.write(full_md)

            print(f"✅ Wrote {output_path} ({len(chunks)} chunks)")

        # Save all files concurrently
        tasks = [
            save_single_file(pdf_name, chunks)
            for pdf_name, chunks in results_by_pdf.items()
        ]
        await asyncio.gather(*tasks)

    async def process_folder_async(self, folder_path: str, output_dir: str):
        """Main async entry point to process all PDFs in a folder."""
        print(f"🔍 Scanning folder: {folder_path}")
        pdf_files = self.get_pdf_files(folder_path) # type: ignore
        print(f"📄 Found {len(pdf_files)} PDF files")

        # Split all PDFs into chunks (CPU-bound, keep synchronous)
        all_chunks = []
        for pdf_path in pdf_files:
            print(f"🔧 Splitting {Path(pdf_path).name} into chunks...")
            chunks = self.split_pdf_to_chunks(pdf_path, PAGES_PER_CHUNK)
            all_chunks.extend(chunks)

        print(f"📦 Total chunks to process: {len(all_chunks)}")
        print(f"🚀 Processing with {MAX_CONCURRENT} concurrent coroutines...")

        # Process all chunks asynchronously
        results_by_pdf = await self.process_chunks_batch(all_chunks)

        # Save results asynchronously
        await self.save_markdown_files_async(results_by_pdf, output_dir)

        print(f"🎉 Completed processing {len(pdf_files)} PDFs!")
        print(f"📁 Output saved to: {output_dir}")

async def main_async():
    """Async main function for use when event loop is already running."""
    api_key = OPEN_AI_API_KEY
    processor = AsyncPDFProcessor(api_key)
    await processor.process_folder_async(FOLDER_PATH, OUTPUT_DIR)

def main():
    """Synchronous main function for standalone execution."""
    api_key = OPEN_AI_API_KEY
    processor = AsyncPDFProcessor(api_key)

    try:
        # Try to run normally (for standalone scripts)
        asyncio.run(processor.process_folder_async(FOLDER_PATH, OUTPUT_DIR))
    except RuntimeError as e:
        if "cannot be called from a running event loop" in str(e):
            print("Event loop already running. Use await main_async() instead.")
            print("Or use: asyncio.create_task(main_async())")
        else:
            raise

# For standalone execution
if __name__ == "__main__":
    await main_async()

# For Jupyter/existing event loop environments, use one of these:
# Option 1: await main_async()
# Option 2: task = asyncio.create_task(main_async())
# Option 3: asyncio.ensure_future(main_async())

🔍 Scanning folder: pdfs_20_2
📄 Found 270 PDF files
🔧 Splitting 01_gcf-b42-02-add17-funding-proposal-package-fp275.pdf into chunks...


/home/vscode/.local/lib/python3.11/site-packages/pypdf/_utils.py:262: RuntimeWarning: coroutine 'AsyncPDFProcessor.process_folder_async' was never awaited
  m = regex.search(name + tok)


🔧 Splitting 02_gcf-b42-02-add16-funding-proposal-package-fp274.pdf into chunks...
🔧 Splitting 03_gcf-b42-02-add15-funding-proposal-package-fp273.pdf into chunks...
🔧 Splitting 04_gcf-b42-02-add14-funding-proposal-package-fp272_0.pdf into chunks...
🔧 Splitting 05_gcf-b42-02-add13-funding-proposal-package-fp271.pdf into chunks...
🔧 Splitting 06_gcf-b42-02-add12-funding-proposal-package-fp270.pdf into chunks...
🔧 Splitting 07_gcf-b42-02-add11-funding-proposal-package-fp269.pdf into chunks...
🔧 Splitting 08_gcf-b42-02-add10-funding-proposal-package-fp268.pdf into chunks...
🔧 Splitting 09_gcf-b42-02-add09-funding-proposal-package-fp267.pdf into chunks...
🔧 Splitting 100_gcf-b30-02-add07.pdf into chunks...
🔧 Splitting 101_gcf-b30-02-add06.pdf into chunks...
🔧 Splitting 102_gcf-b30-02-add05.pdf into chunks...
🔧 Splitting 103_gcf-b30-03-add04.pdf into chunks...
🔧 Splitting 104_gcf-b30-03-add03.pdf into chunks...
🔧 Splitting 105_gcf-b30-02-add02.pdf into chunks...
🔧 Splitting 106_gcf-b30-02-add

CancelledError: 

In [ ]:
#!/usr/bin/env python3
import os
import tempfile
import asyncio
from pathlib import Path
from typing import List, Tuple, Dict
from dataclasses import dataclass
import aiohttp
import json
import random
from enum import Enum

try:
    from pypdf import PdfReader, PdfWriter  # pip install pypdf
except ImportError:
    raise SystemExit("Please: pip install pypdf")

try:
    import aiofiles  # pip install aiofiles
except ImportError:
    raise SystemExit("Please: pip install aiofiles")

# ------------- CONFIG -------------
class ModelConfig(Enum):
    GPT_4O = "gpt-4o"
    GPT_4O_MINI = "gpt-4o-mini"
    O4_MINI ="o4-mini"

class ModelConfigOutPutFolder(Enum):
    GPT_4O_OUPUT = "gpt_4o_output_md"
    GPT_4O_MINI_OUTPUT = "gpt_4o_mini_output_md"
    O4_MINI_OUTPUT ="o4_mini_output_md"

MODEL = ModelConfig.O4_MINI.value        # or "gpt-4o-mini"
FOLDER_PATH = "pdfs_20_2"   # Folder containing PDFs
OUTPUT_DIR = ModelConfigOutPutFolder.O4_MINI_OUTPUT.value  # Output directory for markdown files
PAGES_PER_CHUNK = 3         # lower if you hit context limits
TEMPERATURE = 0.0
MAX_CONCURRENT = 3          # concurrency window per-PDF
# ----------------------------------

OPENAI_API_KEY = "REDACTED-OPENAI-KEY"


@dataclass
class ChunkInfo:
    pdf_name: str
    chunk_index: int
    chunk_path: str
    page_range_label: str


class AsyncPDFProcessor:
    def __init__(self, api_key: str):
        if not api_key:
            raise SystemExit("Set OPENAI_API_KEY in your environment.")
        self.api_key = api_key
        self.base_url = "https://api.openai.com/v1"

    # ---------- FS & splitting ----------

    def get_pdf_files(self, folder_path: str,f_limit: int = 1) -> List[Path]:
        folder = Path(folder_path)
        if not folder.exists():
            raise SystemExit(f"Folder {folder_path} does not exist.")

        pdf_files = list(folder.glob("*.pdf"))[:f_limit]

        if not pdf_files:
            raise SystemExit(f"No PDF files found in {folder_path}")

        return pdf_files

    def split_pdf_to_chunks(self, pdf_path: str, pages_per_chunk: int) -> List[ChunkInfo]:
        reader = PdfReader(pdf_path)
        n_pages = len(reader.pages)
        chunks: List[ChunkInfo] = []
        pdf_name = Path(pdf_path).stem

        for chunk_idx, start in enumerate(range(0, n_pages, pages_per_chunk)):
            end = min(start + pages_per_chunk, n_pages)
            writer = PdfWriter()
            for i in range(start, end):
                writer.add_page(reader.pages[i])

            fd, tmp_pdf = tempfile.mkstemp(
                suffix=f".{pdf_name}.chunk{chunk_idx:03d}.{start+1}-{end}.pdf"
            )
            os.close(fd)
            with open(tmp_pdf, "wb") as f:
                writer.write(f)

            page_range_label = f"pages {start+1}–{end}"
            chunks.append(ChunkInfo(
                pdf_name=pdf_name,
                chunk_index=chunk_idx,
                chunk_path=tmp_pdf,
                page_range_label=page_range_label
            ))
        return chunks

    # ---------- OpenAI helpers ----------

    async def _with_retries(self, coro_factory, *, tries=5, base=0.7, max_sleep=10.0):
        last = None
        for i in range(tries):
            try:
                return await coro_factory()
            except Exception as e:
                last = e
                msg = str(e).lower()
                retryable = any(s in msg for s in ("429", "timeout", "temporarily", "5", "rate"))
                if retryable and i < tries - 1:
                    sleep_for = min(max_sleep, base * (2 ** i)) + random.random()
                    await asyncio.sleep(sleep_for)
                else:
                    break
        raise last

    async def upload_file_async(self, session: aiohttp.ClientSession, file_path: str) -> str:
        headers = {"Authorization": f"Bearer {self.api_key}"}
        # Use a sync handle inside aiohttp FormData (works fine)
        data = aiohttp.FormData()
        data.add_field('purpose', 'assistants')
        data.add_field('file', open(file_path, "rb"), filename=os.path.basename(file_path))
        async with session.post(f"{self.base_url}/files", data=data, headers=headers) as response:
            if response.status != 200:
                error_text = await response.text()
                raise Exception(f"File upload failed: {response.status} - {error_text}")
            result = await response.json()
            return result['id']

    async def delete_file_async(self, session: aiohttp.ClientSession, file_id: str):
        headers = {"Authorization": f"Bearer {self.api_key}"}
        async with session.delete(f"{self.base_url}/files/{file_id}", headers=headers) as _:
            pass  # ignore response

    async def convert_chunk_async(self, session: aiohttp.ClientSession, file_id: str, chunk_info: ChunkInfo,is_temperature_supported:bool=False) -> str:
        headers = {"Authorization": f"Bearer {self.api_key}", "Content-Type": "application/json"}
        payload = {
            "model": MODEL,
            "input": [
                {
                    "role": "system",
                    "content": (
                        "You convert PDFs into clean, GitHub-flavored Markdown for RAG. "
                        "Rules:\n"
                        "1) Preserve structure with proper #/##/### headings.\n"
                        "2) Convert tables to Markdown tables that should be well formated.\n"
                        "3) Keep lists and links.\n"
                        "4) Remove repeated headers/footers and OCR noise.\n"
                        "5) No summaries—output the full text you can read.\n"
                        "6) Output ONLY Markdown (no extra commentary)."
                        "7) Keep the page number of each document."
                    ),
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "input_text", "text": f"Convert this PDF chunk ({chunk_info.page_range_label}) to Markdown."},
                        {"type": "input_file", "file_id": file_id},
                    ],
                },
            ],

        }

        if is_temperature_supported:
            payload["temperature"] = TEMPERATURE

        async with session.post(f"{self.base_url}/responses", json=payload, headers=headers) as response:
            if response.status != 200:
                error_text = await response.text()
                raise Exception(f"API request failed: {response.status} - {error_text}")
            result = await response.json()
            return self.extract_output_text_from_response(result)

    @staticmethod
    def extract_output_text_from_response(r: dict) -> str:
        # Fast-path
        if isinstance(r, dict) and r.get("output_text"):
            return r["output_text"]
        # Generic walk
        out = []
        for o in r.get("output", []):
            for c in o.get("content", []):
                if "text" in c and isinstance(c["text"], str):
                    out.append(c["text"])
                elif c.get("type") in ("output_text", "text"):
                    out.append(c.get("text", ""))
        return "".join(out).strip()

    # ---------- per-PDF parallel window ----------

    async def process_one_chunk(
        self,
        session: aiohttp.ClientSession,
        semaphore: asyncio.Semaphore,
        chunk: ChunkInfo
    ) -> Tuple[int, str]:
        async with semaphore:
            file_id = None
            try:
                # upload with retries
                file_id = await self._with_retries(lambda: self.upload_file_async(session, chunk.chunk_path))
                # convert with retries
                md = await self._with_retries(lambda: self.convert_chunk_async(session, file_id, chunk))
                formatted = f"\n\n---\n*Chunk {chunk.page_range_label}*\n---\n\n{md.strip()}\n"
                print(f"✅ {chunk.pdf_name} | {chunk.page_range_label}")
                return chunk.chunk_index, formatted
            except Exception as e:
                print(f"⚠️ {chunk.pdf_name} | {chunk.page_range_label} failed: {e}")
                return chunk.chunk_index, f"\n\n---\n*Error processing {chunk.page_range_label}: {e}*\n---\n\n"
            finally:
                # delete remote file to keep account tidy
                if file_id:
                    try:
                        await self.delete_file_async(session, file_id)
                    except Exception:
                        pass
                # local temp cleanup
                try:
                    os.remove(chunk.chunk_path)
                except OSError:
                    pass

    async def process_single_pdf(self, session: aiohttp.ClientSession, pdf_path: str, output_dir: str):
        pdf_name = Path(pdf_path).stem
        print(f"\n📄 Processing PDF: {Path(pdf_path).name}")
        chunks = self.split_pdf_to_chunks(pdf_path, PAGES_PER_CHUNK)
        print(f"🔧 Split into {len(chunks)} chunks (window={MAX_CONCURRENT})")

        semaphore = asyncio.Semaphore(MAX_CONCURRENT)

        # Launch all chunk tasks at once; the semaphore enforces the concurrency window.
        tasks = [asyncio.create_task(self.process_one_chunk(session, semaphore, ch)) for ch in chunks]

        # Wait for all to complete
        results: List[Tuple[int, str]] = await asyncio.gather(*tasks)

        # Merge by original order
        results.sort(key=lambda x: x[0])
        merged_md = "".join(md for _, md in results).strip() + "\n"

        # Save
        os.makedirs(output_dir, exist_ok=True)
        out_path = os.path.join(output_dir, f"{pdf_name}.md")
        async with aiofiles.open(out_path, "w", encoding="utf-8") as f:
            await f.write(merged_md)
        print(f"✅ Wrote {out_path}")

    # ---------- top-level orchestrator ----------

    async def process_folder_sequential(self, folder_path: str, output_dir: str):
        print(f"🔍 Scanning: {folder_path}")
        pdf_files = self.get_pdf_files(folder_path)
        print(f"📚 Found {len(pdf_files)} PDFs")

        connector = aiohttp.TCPConnector(
            limit=MAX_CONCURRENT * 2,
            limit_per_host=MAX_CONCURRENT,
            ttl_dns_cache=300,
            use_dns_cache=True,
        )
        timeout = aiohttp.ClientTimeout(total=900)  # longer total for large docs

        async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
            # IMPORTANT: process PDFs **sequentially**
            for idx, pdf in enumerate(pdf_files, 1):
                print(f"\n🚀 [{idx}/{len(pdf_files)}] Start {Path(pdf).name}")
                await self.process_single_pdf(session, pdf, output_dir)
                print(f"🏁 [{idx}/{len(pdf_files)}] Done {Path(pdf).name}")


async def main_async():
    processor = AsyncPDFProcessor(OPENAI_API_KEY)
    await processor.process_folder_sequential(FOLDER_PATH, OUTPUT_DIR) # type: ignore





if __name__ == "__main__":
    await main_async()


🔍 Scanning: pdfs_20_2
📚 Found 1 PDFs

🚀 [1/1] Start 96_gcf-b30-02-add11.pdf

📄 Processing PDF: 96_gcf-b30-02-add11.pdf
🔧 Split into 52 chunks (window=3)
✅ 96_gcf-b30-02-add11 | pages 1–3
✅ 96_gcf-b30-02-add11 | pages 7–9
✅ 96_gcf-b30-02-add11 | pages 4–6
✅ 96_gcf-b30-02-add11 | pages 10–12
✅ 96_gcf-b30-02-add11 | pages 16–18
✅ 96_gcf-b30-02-add11 | pages 13–15
✅ 96_gcf-b30-02-add11 | pages 25–27
✅ 96_gcf-b30-02-add11 | pages 22–24
✅ 96_gcf-b30-02-add11 | pages 19–21
✅ 96_gcf-b30-02-add11 | pages 28–30
✅ 96_gcf-b30-02-add11 | pages 31–33
✅ 96_gcf-b30-02-add11 | pages 34–36
✅ 96_gcf-b30-02-add11 | pages 37–39
✅ 96_gcf-b30-02-add11 | pages 40–42
✅ 96_gcf-b30-02-add11 | pages 43–45
✅ 96_gcf-b30-02-add11 | pages 46–48
✅ 96_gcf-b30-02-add11 | pages 49–51
✅ 96_gcf-b30-02-add11 | pages 52–54
✅ 96_gcf-b30-02-add11 | pages 55–57
✅ 96_gcf-b30-02-add11 | pages 58–60
✅ 96_gcf-b30-02-add11 | pages 61–63
✅ 96_gcf-b30-02-add11 | pages 64–66
✅ 96_gcf-b30-02-add11 | pages 67–69
✅ 96_gcf-b30-02-add11 | p